# UPI Fraud Ring & Merchant Analytics

## 02 - Data Cleaning & Standardization

### Objective

Clean and standardize the raw UPI transaction, KYC, merchant, and
chargeback datasets to create reliable analytical datasets.

### Cleaning Goals

- Normalize user and merchant identifiers
- Remove exact duplicate records
- Standardize transaction amounts
- Convert mixed timestamp formats
- Validate UTR values
- Standardize transaction statuses
- Clean KYC identity fields
- Standardize merchant MCC and category fields
- Parse and standardize chargeback records
- Preserve important data-quality and fraud-related signals
- Validate relationships between datasets

### Output

The cleaned datasets will be saved in:

`../data/cleaned/`

In [1]:
import pandas as pd
import numpy as np
import re
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
transactions = pd.read_csv(
    "../data/raw/track1_upi_transactions.csv"
)

kyc = pd.read_csv(
    "../data/raw/track1_kyc_records.csv"
)

merchants = pd.read_csv(
    "../data/raw/track1_merchants_master.csv"
)

chargebacks = pd.read_json(
    "../data/raw/track1_chargebacks.json"
)

print("Raw datasets loaded successfully!")
print("Transactions:", transactions.shape)
print("KYC:", kyc.shape)
print("Merchants:", merchants.shape)
print("Chargebacks:", chargebacks.shape)

Raw datasets loaded successfully!
Transactions: (20400, 8)
KYC: (36400, 12)
Merchants: (6210, 11)
Chargebacks: (2884, 13)


In [3]:
transactions_clean = transactions.copy()
kyc_clean = kyc.copy()
merchants_clean = merchants.copy()
chargebacks_clean = chargebacks.copy()

print("Working copies created.")

Working copies created.


## 3. Identifier Normalization

Identifiers across the datasets may contain inconsistent capitalization,
whitespace, or formatting. We standardize the identifiers into a
canonical uppercase format before performing joins and analysis.

The following identifiers are normalized:

- Transaction ID
- User ID
- Merchant ID
- Complaint ID

In [4]:
def normalize_id(value):
    """
    Normalize identifiers by:
    - Converting to string
    - Removing leading/trailing whitespace
    - Converting to uppercase
    - Removing spaces and common separators
    """
    if pd.isna(value):
        return np.nan

    value = str(value).strip().upper()
    value = re.sub(r"[\s\-_]+", "", value)

    return value


# Normalize transaction identifiers
for col in ["txn_id", "user_id", "merchant_id"]:
    transactions_clean[col] = transactions_clean[col].apply(normalize_id)

# Normalize KYC user identifier
kyc_clean["user_id"] = kyc_clean["user_id"].apply(normalize_id)

# Normalize merchant identifier
merchants_clean["merchant_id"] = merchants_clean["merchant_id"].apply(normalize_id)

# Normalize chargeback identifiers
for col in ["complaint_id", "txn_id", "user_id", "merchant_id"]:
    chargebacks_clean[col] = chargebacks_clean[col].apply(normalize_id)

print("Identifier normalization completed successfully!")

Identifier normalization completed successfully!


In [5]:
print("========== TRANSACTIONS ==========")
display(
    transactions_clean[["txn_id", "user_id", "merchant_id"]].head(10)
)

print("========== KYC ==========")
display(
    kyc_clean[["user_id"]].head(10)
)

print("========== MERCHANTS ==========")
display(
    merchants_clean[["merchant_id"]].head(10)
)

print("========== CHARGEBACKS ==========")
display(
    chargebacks_clean[
        ["complaint_id", "txn_id", "user_id", "merchant_id"]
    ].head(10)
)

========== TRANSACTIONS ==========


,txn_id,user_id,merchant_id
0,TXN00011869,USR45826,MCH7045
1,TXN00010383,USR79397,MCH5031
2,TXN00008297,USR87810,MCH9809
3,TXN00006448,USR54287,MCH6928
4,TXN00018792,USR53865,MCH8121
5,TXN00000400,USR90546,MCH6773
6,TXN00015249,USR18691,MCH7856
7,TXN00007121,USR16629,MCH7753
8,TXN00008460,USR20899,MCH9111
9,TXN00002541,USR51755,MCH2949


========== KYC ==========


,user_id
0,USR16112
1,USR17216
2,USR45454
3,USR46189
4,USR85256
5,USR20918
6,USR22494
7,USR34742
8,USR83137
9,USR99161


========== MERCHANTS ==========


,merchant_id
0,MCH2849
1,MCH4314
2,MCH1986
3,MCH3899
4,MCH4859
5,MCH6848
6,MCH9933
7,MCH2637
8,MCH8708
9,MCH2430


========== CHARGEBACKS ==========


,complaint_id,txn_id,user_id,merchant_id
0,CBK0002082,TXN00004325,USR97580,MCH1127
1,CBK0001941,TXN00003720,USR54113,3835
2,CBK0001799,TXN00012539,USR17980,MCH3700
3,CBK0002465,TXN00017802,USR76148,MCH4534
4,CBK0001870,TXN00015944,USR24660,MCH1686
5,CBK0002663,TXN00009741,USR40631,MCH9584
6,CBK0000782,TXN00015086,USR58789,MCH6008
7,CBK0001838,TXN00014426,USR30157,MCH4526
8,CBK0001095,TXN00015316,86750,MCH5980
9,CBK0002541,TXN00002822,USR11987,MCH3992


In [6]:
print(
    "Lowercase transaction IDs:",
    transactions_clean["txn_id"]
    .dropna()
    .str.contains(r"[a-z]", regex=True)
    .sum()
)

print(
    "Lowercase user IDs:",
    transactions_clean["user_id"]
    .dropna()
    .str.contains(r"[a-z]", regex=True)
    .sum()
)

print(
    "Lowercase merchant IDs:",
    transactions_clean["merchant_id"]
    .dropna()
    .str.contains(r"[a-z]", regex=True)
    .sum()
)

Lowercase transaction IDs: 0
Lowercase user IDs: 0
Lowercase merchant IDs: 0


## 4. Exact Duplicate Removal

Exact duplicate records are removed from the working datasets to prevent
duplicate rows from inflating transaction, merchant, KYC, and chargeback
analytics.

Records that share the same identifier but contain different information
are retained because they may represent historical changes or potential
data-quality/anomaly signals.

In [7]:
# Store row counts before duplicate removal
before_counts = {
    "Transactions": len(transactions_clean),
    "KYC": len(kyc_clean),
    "Merchants": len(merchants_clean),
    "Chargebacks": len(chargebacks_clean)
}

# Count exact duplicates
duplicate_counts = {
    "Transactions": transactions_clean.duplicated().sum(),
    "KYC": kyc_clean.duplicated().sum(),
    "Merchants": merchants_clean.duplicated().sum(),
    "Chargebacks": chargebacks_clean.duplicated().sum()
}

print("========== EXACT DUPLICATES ==========")

for dataset, count in duplicate_counts.items():
    print(f"{dataset}: {count}")

# Remove exact duplicate rows
transactions_clean = transactions_clean.drop_duplicates().copy()
kyc_clean = kyc_clean.drop_duplicates().copy()
merchants_clean = merchants_clean.drop_duplicates().copy()
chargebacks_clean = chargebacks_clean.drop_duplicates().copy()

print("\nExact duplicate rows removed successfully!")

========== EXACT DUPLICATES ==========
Transactions: 400
KYC: 278
Merchants: 12
Chargebacks: 84

Exact duplicate rows removed successfully!


In [8]:
after_counts = {
    "Transactions": len(transactions_clean),
    "KYC": len(kyc_clean),
    "Merchants": len(merchants_clean),
    "Chargebacks": len(chargebacks_clean)
}

duplicate_summary = pd.DataFrame({
    "Dataset": before_counts.keys(),
    "Rows_Before": before_counts.values(),
    "Duplicates_Removed": duplicate_counts.values(),
    "Rows_After": after_counts.values()
})

display(duplicate_summary)

,Dataset,Rows_Before,Duplicates_Removed,Rows_After
0,Transactions,20400,400,20000
1,KYC,36400,278,36122
2,Merchants,6210,12,6198
3,Chargebacks,2884,84,2800


In [9]:
print("========== DUPLICATE CHECK AFTER CLEANING ==========")

print("Transactions:", transactions_clean.duplicated().sum())
print("KYC:", kyc_clean.duplicated().sum())
print("Merchants:", merchants_clean.duplicated().sum())
print("Chargebacks:", chargebacks_clean.duplicated().sum())

========== DUPLICATE CHECK AFTER CLEANING ==========
Transactions: 0
KYC: 0
Merchants: 0
Chargebacks: 0


## 5. Transaction Amount Cleaning

Transaction amounts are converted into a consistent numeric format.
Currency symbols, commas, whitespace, and other non-numeric formatting
are removed where applicable.

Negative transaction amounts are flagged for investigation rather than
silently deleted, as they may represent data-quality anomalies.

In [10]:
# Convert transaction amounts to numeric values

def clean_amount(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    # Remove currency symbols and commas
    value = value.replace("₹", "")
    value = value.replace("Rs.", "")
    value = value.replace("Rs", "")
    value = value.replace(",", "")
    value = value.strip()

    # Convert to numeric
    return pd.to_numeric(value, errors="coerce")


transactions_clean["amount"] = transactions_clean["amount"].apply(clean_amount)

print("Amount cleaning completed successfully!")
print("Amount data type:", transactions_clean["amount"].dtype)
print("Missing amounts:", transactions_clean["amount"].isna().sum())

Amount cleaning completed successfully!
Amount data type: float64
Missing amounts: 2001


In [11]:
print("========== TRANSACTION AMOUNT SUMMARY ==========")

display(
    transactions_clean["amount"].describe()
)

========== TRANSACTION AMOUNT SUMMARY ==========


count    17999.00000
mean     11892.75367
std       8139.05857
min     -24847.86000
25%       5806.62500
50%      12200.46000
75%      18548.49500
max      24998.12000
Name: amount, dtype: float64

In [12]:
negative_amounts = (transactions_clean["amount"] < 0).sum()

print("Negative transaction amounts:", negative_amounts)

Negative transaction amounts: 420


## 6. Timestamp Normalization

Transaction timestamps may contain different date/time formats.
They are converted into a consistent datetime representation for
time-based fraud and merchant analytics.

Invalid or unparseable timestamps are retained as missing values
and will be flagged during data-quality validation.

In [13]:
# Timestamp normalization

transactions_clean["timestamp_original"] = transactions_clean["timestamp"]

transactions_clean["timestamp"] = pd.to_datetime(
    transactions_clean["timestamp"],
    errors="coerce"
)

print("Timestamp normalization completed successfully!")
print("Timestamp data type:", transactions_clean["timestamp"].dtype)
print("Missing/invalid timestamps:", transactions_clean["timestamp"].isna().sum())

Timestamp normalization completed successfully!
Timestamp data type: datetime64[us]
Missing/invalid timestamps: 8933


In [14]:
print("========== TIMESTAMP RANGE ==========")

print("Earliest transaction:", transactions_clean["timestamp"].min())
print("Latest transaction:", transactions_clean["timestamp"].max())

========== TIMESTAMP RANGE ==========
Earliest transaction: 2026-01-01 00:12:54
Latest transaction: 2026-03-31 23:53:53


In [15]:
invalid_timestamps = transactions_clean["timestamp"].isna().sum()

print("Invalid/unparseable timestamps:", invalid_timestamps)

Invalid/unparseable timestamps: 8933


In [16]:
# Create time-based features for later fraud and merchant analysis

transactions_clean["transaction_date"] = transactions_clean["timestamp"].dt.date
transactions_clean["transaction_hour"] = transactions_clean["timestamp"].dt.hour
transactions_clean["transaction_day"] = transactions_clean["timestamp"].dt.day_name()
transactions_clean["transaction_month"] = transactions_clean["timestamp"].dt.month
transactions_clean["transaction_year"] = transactions_clean["timestamp"].dt.year

print("Time-based features created successfully!")

Time-based features created successfully!


In [17]:
# Robust timestamp normalization for mixed formats

def normalize_timestamp(value):
    if pd.isna(value):
        return pd.NaT

    value_str = str(value).strip()

    if value_str == "":
        return pd.NaT

    # Try numeric epoch timestamps
    try:
        numeric_value = float(value_str)

        # Epoch milliseconds
        if numeric_value >= 1_000_000_000_000:
            return pd.to_datetime(
                numeric_value,
                unit="ms",
                errors="coerce"
            )

        # Epoch seconds
        elif numeric_value >= 1_000_000_000:
            return pd.to_datetime(
                numeric_value,
                unit="s",
                errors="coerce"
            )
    except:
        pass

    # Try normal/mixed date-time formats
    return pd.to_datetime(
        value_str,
        format="mixed",
        errors="coerce"
    )


transactions_clean["timestamp"] = (
    transactions_clean["timestamp_original"]
    .apply(normalize_timestamp)
)

print("Mixed timestamp normalization completed!")
print("Timestamp data type:", transactions_clean["timestamp"].dtype)
print("Invalid timestamps:", transactions_clean["timestamp"].isna().sum())

Mixed timestamp normalization completed!
Timestamp data type: datetime64[us]
Invalid timestamps: 0


In [18]:
print("========== TIMESTAMP CHECK ==========")

print(
    "Valid timestamps:",
    transactions_clean["timestamp"].notna().sum()
)

print(
    "Invalid timestamps:",
    transactions_clean["timestamp"].isna().sum()
)

print(
    "Earliest transaction:",
    transactions_clean["timestamp"].min()
)

print(
    "Latest transaction:",
    transactions_clean["timestamp"].max()
)

========== TIMESTAMP CHECK ==========
Valid timestamps: 20000
Invalid timestamps: 0
Earliest transaction: 2026-01-01 00:00:00
Latest transaction: 2026-12-03 23:09:05


In [19]:
invalid_mask = transactions_clean["timestamp"].isna()

display(
    transactions_clean.loc[
        invalid_mask,
        ["timestamp_original", "txn_id"]
    ].head(20)
)

,timestamp_original,txn_id


In [20]:
print("========== TRANSACTION DATE DISTRIBUTION ==========")

display(
    transactions_clean["timestamp"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

========== TRANSACTION DATE DISTRIBUTION ==========


timestamp
2026-01    6478
2026-02    5823
2026-03    6467
2026-04     129
2026-05     138
2026-06     137
2026-07     124
2026-08     148
2026-09     135
2026-10     141
2026-11     134
2026-12     146
Freq: M, Name: count, dtype: int64

In [21]:
transactions_clean["transaction_date"] = transactions_clean["timestamp"].dt.date
transactions_clean["transaction_hour"] = transactions_clean["timestamp"].dt.hour
transactions_clean["transaction_day"] = transactions_clean["timestamp"].dt.day_name()
transactions_clean["transaction_month"] = transactions_clean["timestamp"].dt.month
transactions_clean["transaction_year"] = transactions_clean["timestamp"].dt.year

print("Time-based features created successfully!")

Time-based features created successfully!


## 7. UTR Validation

UPI transaction reference numbers (UTRs) are standardized and checked
for missing, empty, or malformed values.

Invalid UTRs are flagged rather than removed because they may be useful
for identifying transaction-level data-quality anomalies.

In [22]:
# Standardize UTR values

transactions_clean["utr"] = (
    transactions_clean["utr"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Treat empty strings as missing
transactions_clean["utr"] = transactions_clean["utr"].replace(
    ["", "NAN", "NONE", "NULL"],
    pd.NA
)

print("UTR standardization completed!")
print("Missing UTRs:", transactions_clean["utr"].isna().sum())

UTR standardization completed!
Missing UTRs: 1000


In [23]:
print("========== UTR CHECK ==========")

print("Total transactions:", len(transactions_clean))
print("Valid/non-missing UTRs:", transactions_clean["utr"].notna().sum())
print("Missing UTRs:", transactions_clean["utr"].isna().sum())

print("\nSample UTR values:")
display(transactions_clean[["txn_id", "utr"]].head(10))

========== UTR CHECK ==========
Total transactions: 20000
Valid/non-missing UTRs: 19000
Missing UTRs: 1000

Sample UTR values:


,txn_id,utr
0,TXN00011869,UTR6498104698
1,TXN00010383,UTR7656190355
2,TXN00008297,UTR9037001889
3,TXN00006448,UTR5257823698
4,TXN00018792,UTR4204272894
5,TXN00000400,UTR2080442730
6,TXN00015249,UTR5471158093
7,TXN00007121,UTR3330570580
8,TXN00008460,UTR6505850115
9,TXN00002541,UTR 2787678319


## 8. MCC and Category Cleaning

Merchant Category Codes (MCCs) are standardized into a consistent
numeric representation.

Missing or malformed MCC values are retained and flagged rather than
deleted because missing category information can itself be useful for
merchant-risk and fraud analysis.

In [24]:
# Standardize MCC values

transactions_clean["mcc_original"] = transactions_clean["mcc"]

transactions_clean["mcc"] = (
    transactions_clean["mcc"]
    .astype("string")
    .str.strip()
)

# Convert numeric-looking MCC values to numeric
transactions_clean["mcc"] = pd.to_numeric(
    transactions_clean["mcc"],
    errors="coerce"
)

print("MCC standardization completed!")
print("Missing/invalid MCCs:", transactions_clean["mcc"].isna().sum())

MCC standardization completed!
Missing/invalid MCCs: 2872


In [25]:
print("========== MCC CHECK ==========")

print("Total transactions:", len(transactions_clean))
print("Valid MCCs:", transactions_clean["mcc"].notna().sum())
print("Missing/invalid MCCs:", transactions_clean["mcc"].isna().sum())

print("\nMCC data type:", transactions_clean["mcc"].dtype)

print("\nSample MCC values:")
display(
    transactions_clean[["txn_id", "mcc_original", "mcc"]].head(15)
)

========== MCC CHECK ==========
Total transactions: 20000
Valid MCCs: 17128
Missing/invalid MCCs: 2872

MCC data type: Float64

Sample MCC values:


,txn_id,mcc_original,mcc
0,TXN00011869,5411.0,5411.0
1,TXN00010383,4131.0,4131.0
2,TXN00008297,5411.0,5411.0
3,TXN00006448,5411.0,5411.0
4,TXN00018792,4131.0,4131.0
5,TXN00000400,5812.0,5812.0
6,TXN00015249,NaN,<NA>
7,TXN00007121,5411.0,5411.0
8,TXN00008460,NaN,<NA>
9,TXN00002541,5411.0,5411.0


In [26]:
print("========== TOP MCC VALUES ==========")

display(
    transactions_clean["mcc"]
    .value_counts(dropna=False)
    .head(20)
)

========== TOP MCC VALUES ==========


mcc
5411.0    5720
5912.0    2891
<NA>      2872
5812.0    2866
7011.0    2827
4131.0    2824
Name: count, dtype: Int64

In [27]:
transactions_clean["mcc_missing_flag"] = (
    transactions_clean["mcc"].isna()
)

print(
    "Transactions with missing/invalid MCC:",
    transactions_clean["mcc_missing_flag"].sum()
)

Transactions with missing/invalid MCC: 2872


In [28]:
print("========== TRANSACTION CLEANING CHECKPOINT ==========")

print("Rows:", len(transactions_clean))
print("Columns:", len(transactions_clean.columns))

print("\nMissing values:")
display(
    transactions_clean[
        ["txn_id", "user_id", "merchant_id", "amount",
         "timestamp", "utr", "mcc", "status"]
    ].isna().sum()
)

========== TRANSACTION CLEANING CHECKPOINT ==========
Rows: 20000
Columns: 16

Missing values:


txn_id            0
user_id           0
merchant_id       0
amount         2001
timestamp         0
utr            1000
mcc            2872
status            0
dtype: int64

## 9. KYC Data Cleaning

The KYC dataset is standardized to support customer profiling,
identity-anomaly detection, and joins with UPI transactions.

Sensitive identity fields are normalized for consistency while
preserving missing and conflicting values as analytical signals.

In [29]:
print("========== KYC DATASET ==========")

print("Rows:", len(kyc_clean))
print("Columns:", len(kyc_clean.columns))

print("\nKYC columns:")
print(kyc_clean.columns.tolist())

print("\nSample records:")
display(kyc_clean.head(10))

========== KYC DATASET ==========
Rows: 36122
Columns: 12

KYC columns:
['user_id', 'full_name', 'pan', 'aadhaar', 'date_of_birth', 'city', 'state', 'monthly_income', 'occupation', 'signup_timestamp', 'kyc_status', 'risk_segment']

Sample records:


,user_id,full_name,pan,aadhaar,date_of_birth,city,state,monthly_income,occupation,signup_timestamp,kyc_status,risk_segment
0,USR16112,Dhriti Deshmukh,SEJAA8194O,715658320763,06/04/1967 12:14 AM,Bombay,Maharashtra,35119,Retired,2025-12-02 02:18:16,Done,LOW
1,USR17216,Megha Jani,QT0ZZ5561X,023412025028,NaN,Lucknow,Uttar Pradesh,80907,Freelancer,01-31-2024,Verified,medium
2,USR45454,PANINI LAL,QCNTL9489O,2781 6299 9816,NaN,kolkata,West Bengal,"₹11,214",Farmer,2026-01-30 23:57:18,Pending,High
3,USR46189,Jack Parikh,RRVUI4059Q,0667 8032 7731,NaN,Amritsar,Punjab,27.3k,Student,09-04-2025,VERIFIED,low
4,USR85256,Eta Ravi,ygvoi9236w,176258817110,1969-02-19 08:25:23,delhi,Delhi,"INR 24,705",Salaried,03-Feb-2026,APPROVED,LOW
5,USR20918,Vanya Chad,jdgoh0074y,894463789078,20/04/1961,BLR,Karnataka,14124,Self Employed,25/04/2024,REJECTED,MEDIUM
6,USR22494,NIRJA SHETTY,JWZVO4240H,5277136337802,05-Sep-1990,Chennai,Tamil Nadu,16160,Retired,NaN,Reject,medium
7,USR34742,Ati Dash,OOQCG9609N,985048780290,12-20-1962,Jaipur,Rajasthan,43738,Unemployed,07-25-2025,APPROVED,high
8,USR83137,ria kale,NaN,4902616251043,09-Sep-2005,Jaipur,Rajasthan,25151,Freelancer,03/04/2024,Pending,low
9,USR99161,jalsa malhotra,QLSKS5105S,944300371193,20/05/1963 05:27 AM,Jalandhar,Punjab,-8083,Retired,NaN,Rejected,Low


In [30]:
print("========== KYC MISSING VALUES ==========")

kyc_missing = kyc_clean.isna().sum()

display(
    kyc_missing[kyc_missing > 0]
    .sort_values(ascending=False)
)

========== KYC MISSING VALUES ==========


date_of_birth       2921
monthly_income      2906
signup_timestamp    2886
aadhaar             2641
pan                 1883
dtype: int64

In [31]:
# Standardize KYC text fields

text_columns = [
    "full_name",
    "city",
    "state",
    "occupation",
    "kyc_status",
    "risk_segment"
]

for col in text_columns:
    if col in kyc_clean.columns:
        kyc_clean[col] = (
            kyc_clean[col]
            .astype("string")
            .str.strip()
            .str.upper()
        )

print("KYC text fields standardized successfully!")

KYC text fields standardized successfully!


In [32]:
# Standardize PAN

if "pan" in kyc_clean.columns:
    kyc_clean["pan"] = (
        kyc_clean["pan"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    kyc_clean["pan"] = kyc_clean["pan"].replace(
        ["", "NAN", "NONE", "NULL"],
        pd.NA
    )

print("PAN normalization completed.")
print("Missing PAN:", kyc_clean["pan"].isna().sum())

PAN normalization completed.
Missing PAN: 1883


In [33]:
# Standardize Aadhaar

if "aadhaar" in kyc_clean.columns:
    kyc_clean["aadhaar"] = (
        kyc_clean["aadhaar"]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.strip()
    )

    kyc_clean["aadhaar"] = kyc_clean["aadhaar"].replace(
        ["", "NAN", "NONE", "NULL"],
        pd.NA
    )

print("Aadhaar normalization completed.")
print("Missing Aadhaar:", kyc_clean["aadhaar"].isna().sum())

Aadhaar normalization completed.
Missing Aadhaar: 2641


In [34]:
print("========== KYC STATUS ==========")

display(
    kyc_clean["kyc_status"]
    .value_counts(dropna=False)
)

print("\n========== RISK SEGMENT ==========")

display(
    kyc_clean["risk_segment"]
    .value_counts(dropna=False)
)

========== KYC STATUS ==========


kyc_status
VERIFIED        9377
APPROVED        4673
KYC_DONE        4560
V               4526
DONE            4419
PENDING         2475
REJECTED        1472
P                995
IN_PROGRESS      977
UNDER REVIEW     975
R                607
REJECT           542
FAILED           524
Name: count, dtype: Int64


========== RISK SEGMENT ==========


risk_segment
LOW        20821
MEDIUM      9762
HIGH        3693
UNKNOWN     1846
Name: count, dtype: Int64

In [35]:
print("========== IDENTITY CONFLICT CHECK ==========")

identity_check = (
    kyc_clean
    .groupby("user_id")
    .agg(
        distinct_names=("full_name", "nunique"),
        distinct_pans=("pan", "nunique"),
        distinct_aadhaars=("aadhaar", "nunique")
    )
    .reset_index()
)

identity_check["identity_conflict_flag"] = (
    (identity_check["distinct_names"] > 1) |
    (identity_check["distinct_pans"] > 1) |
    (identity_check["distinct_aadhaars"] > 1)
)

print(
    "Users with potential identity conflicts:",
    identity_check["identity_conflict_flag"].sum()
)

display(
    identity_check[
        identity_check["identity_conflict_flag"]
    ].head(20)
)

========== IDENTITY CONFLICT CHECK ==========
Users with potential identity conflicts: 4999


,user_id,distinct_names,distinct_pans,distinct_aadhaars,identity_conflict_flag
31,11947,2,2,2,True
213,23776,2,1,2,True
243,25581,2,2,2,True
402,35009,2,2,2,True
444,37858,2,2,1,True
472,40148,2,2,2,True
528,43692,2,2,2,True
716,55563,2,2,1,True
914,67282,2,2,1,True
1283,90685,2,2,2,True


In [36]:
print("========== STANDARDIZING KYC STATUS ==========")

status_map = {
    "V": "VERIFIED",
    "VERIFIED": "VERIFIED",

    "APPROVED": "APPROVED",
    "KYC_DONE": "APPROVED",
    "DONE": "APPROVED",

    "P": "PENDING",
    "PENDING": "PENDING",

    "REJECTED": "REJECTED",
    "REJECT": "REJECTED",
    "R": "REJECTED",

    "IN_PROGRESS": "IN_PROGRESS",

    "UNDER_REVIEW": "UNDER_REVIEW",

    "FAILED": "FAILED"
}

kyc_clean["kyc_status_original"] = kyc_clean["kyc_status"]

kyc_clean["kyc_status"] = (
    kyc_clean["kyc_status"]
    .map(status_map)
    .fillna("UNKNOWN")
)

print("KYC status standardization completed!")

print("\nStandardized KYC statuses:")
display(
    kyc_clean["kyc_status"]
    .value_counts(dropna=False)
)

========== STANDARDIZING KYC STATUS ==========
KYC status standardization completed!

Standardized KYC statuses:


kyc_status
VERIFIED       13903
APPROVED       13652
PENDING         3470
REJECTED        2621
IN_PROGRESS      977
UNKNOWN          975
FAILED           524
Name: count, dtype: int64

In [37]:
print("========== KYC RISK FEATURES ==========")

# Missing identity/document indicators
kyc_clean["pan_missing_flag"] = kyc_clean["pan"].isna()
kyc_clean["aadhaar_missing_flag"] = kyc_clean["aadhaar"].isna()

# Missing profile information
kyc_clean["dob_missing_flag"] = kyc_clean["date_of_birth"].isna()
kyc_clean["income_missing_flag"] = kyc_clean["monthly_income"].isna()
kyc_clean["signup_missing_flag"] = kyc_clean["signup_timestamp"].isna()

# KYC status risk
kyc_clean["kyc_rejected_flag"] = (
    kyc_clean["kyc_status"] == "REJECTED"
)

kyc_clean["kyc_pending_flag"] = (
    kyc_clean["kyc_status"].isin(
        ["PENDING", "IN_PROGRESS", "UNDER_REVIEW"]
    )
)

# High-risk segment
kyc_clean["high_risk_segment_flag"] = (
    kyc_clean["risk_segment"] == "HIGH"
)

print("KYC risk indicators created successfully!")

display(
    kyc_clean[
        [
            "user_id",
            "kyc_status",
            "risk_segment",
            "pan_missing_flag",
            "aadhaar_missing_flag",
            "kyc_rejected_flag",
            "high_risk_segment_flag"
        ]
    ].head(10)
)

========== KYC RISK FEATURES ==========
KYC risk indicators created successfully!


,user_id,kyc_status,risk_segment,pan_missing_flag,aadhaar_missing_flag,kyc_rejected_flag,high_risk_segment_flag
0,USR16112,APPROVED,LOW,False,False,False,False
1,USR17216,VERIFIED,MEDIUM,False,False,False,False
2,USR45454,PENDING,HIGH,False,False,False,True
3,USR46189,VERIFIED,LOW,False,False,False,False
4,USR85256,APPROVED,LOW,False,False,False,False
5,USR20918,REJECTED,MEDIUM,False,False,True,False
6,USR22494,REJECTED,MEDIUM,False,False,True,False
7,USR34742,APPROVED,HIGH,False,False,False,True
8,USR83137,PENDING,LOW,True,False,False,False
9,USR99161,REJECTED,LOW,False,False,True,False


In [38]:
print("========== BUILDING IDENTITY PROFILE ==========")

identity_profile = (
    kyc_clean
    .groupby("user_id")
    .agg(
        kyc_records=("user_id", "size"),
        distinct_names=("full_name", "nunique"),
        distinct_pans=("pan", "nunique"),
        distinct_aadhaars=("aadhaar", "nunique"),
        missing_pan_records=("pan_missing_flag", "sum"),
        missing_aadhaar_records=("aadhaar_missing_flag", "sum"),
        rejected_records=("kyc_rejected_flag", "sum"),
        pending_records=("kyc_pending_flag", "sum"),
        high_risk_records=("high_risk_segment_flag", "sum")
    )
    .reset_index()
)

identity_profile["identity_conflict_flag"] = (
    (identity_profile["distinct_names"] > 1) |
    (identity_profile["distinct_pans"] > 1) |
    (identity_profile["distinct_aadhaars"] > 1)
)

identity_profile["identity_conflict_count"] = (
    (identity_profile["distinct_names"] > 1).astype(int) +
    (identity_profile["distinct_pans"] > 1).astype(int) +
    (identity_profile["distinct_aadhaars"] > 1).astype(int)
)

print("Identity profile created!")
print("Unique users:", len(identity_profile))

print(
    "Users with identity conflicts:",
    identity_profile["identity_conflict_flag"].sum()
)

display(
    identity_profile
    .sort_values(
        ["identity_conflict_flag", "identity_conflict_count"],
        ascending=False
    )
    .head(20)
)

========== BUILDING IDENTITY PROFILE ==========
Identity profile created!
Unique users: 29355
Users with identity conflicts: 4999


,user_id,kyc_records,distinct_names,distinct_pans,distinct_aadhaars,missing_pan_records,missing_aadhaar_records,rejected_records,pending_records,high_risk_records,identity_conflict_flag,identity_conflict_count
31,11947,2,2,2,2,0,0,1,1,0,True,3
243,25581,2,2,2,2,0,0,0,0,0,True,3
402,35009,3,2,2,2,0,0,0,2,0,True,3
472,40148,2,2,2,2,0,0,0,0,0,True,3
528,43692,2,2,2,2,0,0,0,0,0,True,3
1283,90685,2,2,2,2,0,0,1,0,0,True,3
1393,97801,2,2,2,2,0,0,0,0,0,True,3
1449,USR10043,2,2,2,2,0,0,0,1,0,True,3
1491,USR10188,3,2,2,2,0,0,1,0,2,True,3
1496,USR10202,2,2,2,2,0,0,0,1,0,True,3


In [39]:
import os

output_dir = r"/mnt/data/upi_fraud_project"

os.makedirs(output_dir, exist_ok=True)

kyc_clean.to_csv(
    os.path.join(output_dir, "kyc_clean.csv"),
    index=False
)

identity_profile.to_csv(
    os.path.join(output_dir, "kyc_identity_profile.csv"),
    index=False
)

print("========== KYC OUTPUTS SAVED ==========")

print("KYC dataset:")
print(os.path.join(output_dir, "kyc_clean.csv"))

print("\nIdentity profile:")
print(os.path.join(output_dir, "kyc_identity_profile.csv"))

========== KYC OUTPUTS SAVED ==========
KYC dataset:
/mnt/data/upi_fraud_project\kyc_clean.csv

Identity profile:
/mnt/data/upi_fraud_project\kyc_identity_profile.csv


In [41]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir())

Current folder:
C:\Users\VIKASH\Desktop\UPI-Fraud-Ring-Merchant-Analytics\notebooks

Files/folders here:
['.ipynb_checkpoints', 'Data_Cleaning.ipynb', 'Data_understanding.ipynb']


In [42]:
print("========== LOADING MERCHANT DATA ==========")

merchant_file = r"../data/raw/track1_merchants_master.csv"

merchants = pd.read_csv(merchant_file)

print("Merchant dataset loaded successfully!")
print("Rows:", len(merchants))
print("Columns:", len(merchants.columns))

print("\nMerchant columns:")
print(merchants.columns.tolist())

print("\nFirst 5 records:")
display(merchants.head())

========== LOADING MERCHANT DATA ==========
Merchant dataset loaded successfully!
Rows: 6210
Columns: 11

Merchant columns:
['merchant_id', 'merchant_name', 'mcc', 'merchant_category', 'business_type', 'city', 'state', 'onboarding_date', 'settlement_account', 'merchant_status', 'declared_avg_ticket_size']

First 5 records:


,merchant_id,merchant_name,mcc,merchant_category,business_type,city,state,onboarding_date,settlement_account,merchant_status,declared_avg_ticket_size
0,mch2849,"BHAVSAR, KOTA AND ZACHARIA",MCC-7011,hotel_lodging,Private Limited,Ludhiana,Punjab,09-23-2025,NaN,Inactive,"INR 2,432.18"
1,MCH4314,Deshmukh Ltd,5699,Apparel,PRIVATE_LIMITED,Jalandhar,Punjab,NaN,3021439858,I,986.05
2,MCH1986,"Bains, Chanda and Gh0sh",5311,Retail,individual,Jalandhar,Punjab,10/01/2026,NaN,A,"INR 1,427.52"
3,mch3899,Goswami-Bath,4131,Transportation,Sole Proprietor,Hyderabad,Telangana,14-Jan-2023,NaN,ACTIVE,-1271.48
4,MCH4859,VASA-RAJU,5812,Restaurant,SOLE_PROPRIETOR,Hyd,Telangana,1742180385,XXXX9523,A,Rs. 214


In [43]:
print("========== MERCHANT DATA QUALITY CHECK ==========")

print("Rows:", len(merchants))
print("Columns:", len(merchants.columns))

print("\n========== MISSING VALUES ==========")

merchant_missing = merchants.isna().sum()

display(
    merchant_missing[merchant_missing > 0]
    .sort_values(ascending=False)
)

print("\n========== DUPLICATE ROWS ==========")

print(
    "Exact duplicate rows:",
    merchants.duplicated().sum()
)

print("\n========== MERCHANT ID CHECK ==========")

print(
    "Unique merchant IDs:",
    merchants["merchant_id"].nunique()
)

print(
    "Missing merchant IDs:",
    merchants["merchant_id"].isna().sum()
)

print("\n========== MCC SAMPLE ==========")

display(
    merchants["mcc"]
    .astype(str)
    .value_counts()
    .head(15)
)

print("\n========== MERCHANT CATEGORY ==========")

display(
    merchants["merchant_category"]
    .astype(str)
    .value_counts()
    .head(15)
)

print("\n========== MERCHANT STATUS ==========")

display(
    merchants["merchant_status"]
    .astype(str)
    .value_counts(dropna=False)
)

print("\n========== BUSINESS TYPE ==========")

display(
    merchants["business_type"]
    .astype(str)
    .value_counts(dropna=False)
    .head(15)
)

========== MERCHANT DATA QUALITY CHECK ==========
Rows: 6210
Columns: 11

========== MISSING VALUES ==========


settlement_account          2451
mcc                          514
onboarding_date              499
declared_avg_ticket_size     371
dtype: int64


========== DUPLICATE ROWS ==========
Exact duplicate rows: 12

========== MERCHANT ID CHECK ==========
Unique merchant IDs: 5083
Missing merchant IDs: 0

========== MCC SAMPLE ==========


mcc
4131       435
7011       428
4814       419
5311       418
5699       410
5812       405
5411       402
5912       390
5999       381
5942       374
misc        99
UNKNOWN     80
05912       68
05699       68
05812       66
Name: count, dtype: int64


========== MERCHANT CATEGORY ==========


merchant_category
Department Store    168
HOTEL_LODGING       167
Retail              164
phone service       161
Telecom             159
Stationery          156
CLOTHS              154
Mobile Recharge     152
Hotel               151
DEPT_STORE          150
Miscellaneous       150
TELECOM             148
Retail Other        146
Transport           146
Books               144
Name: count, dtype: int64


========== MERCHANT STATUS ==========


merchant_status
Active       1084
Enabled      1037
Live          984
ACTIVE        971
A             961
Inactive      200
Disabled      147
INACTIVE      139
Closed        137
I             136
Suspended     136
SUSPENDED      80
Hold           80
S              70
Blocked        48
Name: count, dtype: int64


========== BUSINESS TYPE ==========


business_type
INDIVIDUAL         789
PARTNERSHIP        719
individual         417
SOLE_PROPRIETOR    415
partnership        411
PRIVATE-LIMITED    408
PRIVATE_LIMITED    394
Individual         390
sole_proprietor    388
Partnership        381
SOLE-PROPRIETOR    377
private_limited    377
Sole Proprietor    376
Private Limited    368
Name: count, dtype: int64

In [44]:
print("========== STANDARDIZING MERCHANT DATA ==========")

# --------------------------------------------------
# 1. Standardize Merchant ID
# --------------------------------------------------

merchants["merchant_id"] = (
    merchants["merchant_id"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"[^A-Z0-9]", "", regex=True)
)

print("Merchant ID standardization completed.")

# --------------------------------------------------
# 2. Standardize Merchant Name
# --------------------------------------------------

if "merchant_name" in merchants.columns:
    merchants["merchant_name"] = (
        merchants["merchant_name"]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# --------------------------------------------------
# 3. Standardize Merchant Category
# --------------------------------------------------

if "merchant_category" in merchants.columns:
    merchants["merchant_category"] = (
        merchants["merchant_category"]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"[^A-Z0-9]+", "_", regex=True)
        .str.strip("_")
    )

print("Merchant category standardized.")

# --------------------------------------------------
# 4. Standardize Business Type
# --------------------------------------------------

if "business_type" in merchants.columns:
    merchants["business_type"] = (
        merchants["business_type"]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"[^A-Z0-9]+", "_", regex=True)
        .str.strip("_")
    )

print("Business type standardized.")

# --------------------------------------------------
# 5. Standardize City
# --------------------------------------------------

if "city" in merchants.columns:
    merchants["city"] = (
        merchants["city"]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
    )

# --------------------------------------------------
# 6. Standardize State
# --------------------------------------------------

if "state" in merchants.columns:
    merchants["state"] = (
        merchants["state"]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
    )

print("Location fields standardized.")

# --------------------------------------------------
# Preview
# --------------------------------------------------

print("\n========== STANDARDIZED MERCHANT SAMPLE ==========")

display(
    merchants[
        [
            "merchant_id",
            "merchant_name",
            "mcc",
            "merchant_category",
            "business_type",
            "city",
            "state"
        ]
    ].head(10)
)

========== STANDARDIZING MERCHANT DATA ==========
Merchant ID standardization completed.
Merchant category standardized.
Business type standardized.
Location fields standardized.

========== STANDARDIZED MERCHANT SAMPLE ==========


,merchant_id,merchant_name,mcc,merchant_category,business_type,city,state
0,MCH2849,"BHAVSAR, KOTA AND ZACHARIA",MCC-7011,HOTEL_LODGING,PRIVATE_LIMITED,LUDHIANA,PUNJAB
1,MCH4314,Deshmukh Ltd,5699,APPAREL,PRIVATE_LIMITED,JALANDHAR,PUNJAB
2,MCH1986,"Bains, Chanda and Gh0sh",5311,RETAIL,INDIVIDUAL,JALANDHAR,PUNJAB
3,MCH3899,Goswami-Bath,4131,TRANSPORTATION,SOLE_PROPRIETOR,HYDERABAD,TELANGANA
4,MCH4859,VASA-RAJU,5812,RESTAURANT,SOLE_PROPRIETOR,HYD,TELANGANA
5,MCH6848,Dora LLC,5311,DEPARTMENT_STORE,PARTNERSHIP,CHENNAI,TAMIL NADU
6,MCH9933,Bali-Zachariah,5411,GROCERY_STORES,SOLE_PROPRIETOR,JPR,RAJASTHAN
7,MCH2637,Joshi Group,5999,MISC_RETAIL,PARTNERSHIP,CHENNAI,TAMIL NADU
8,MCH8708,"Mukherjee, Sami and Purohit",5999,RETAIL_OTHER,SOLE_PROPRIETOR,HYDERABAD,TELANGANA
9,MCH2430,Bora Group,4814,TELECOM,INDIVIDUAL,AMRITSAR,PUNJAB


In [45]:
print("========== CLEANING MCC ==========")

# Convert MCC to string
merchants["mcc"] = (
    merchants["mcc"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Remove common prefixes such as MCC-
merchants["mcc"] = (
    merchants["mcc"]
    .str.replace(r"^MCC[-_\s]*", "", regex=True)
)

# Keep only numeric MCC values
mcc_numeric = pd.to_numeric(
    merchants["mcc"],
    errors="coerce"
)

# Valid MCC should be a 4-digit numeric code
merchants["mcc_valid_flag"] = (
    mcc_numeric.between(1000, 9999)
)

# Replace invalid/missing MCC with UNKNOWN
merchants.loc[
    ~merchants["mcc_valid_flag"],
    "mcc"
] = "UNKNOWN"

# Make valid MCC codes consistent as 4-digit strings
valid_mask = merchants["mcc"] != "UNKNOWN"

merchants.loc[valid_mask, "mcc"] = (
    pd.to_numeric(
        merchants.loc[valid_mask, "mcc"],
        errors="coerce"
    )
    .astype("Int64")
    .astype("string")
)

print("MCC cleaning completed.")

print("\nMCC status:")
print(
    merchants["mcc_valid_flag"]
    .value_counts(dropna=False)
)

print("\nCleaned MCC distribution:")
display(
    merchants["mcc"]
    .value_counts(dropna=False)
    .head(20)
)

========== CLEANING MCC ==========
MCC cleaning completed.

MCC status:
mcc_valid_flag
True    5517
<NA>     693
Name: count, dtype: Int64

Cleaned MCC distribution:


mcc
<NA>       613
4131       589
5311       580
5699       571
7011       557
4814       557
5912       550
5812       547
5411       531
5999       522
5942       513
UNKNOWN     80
Name: count, dtype: Int64

In [46]:
print("========== FIXING REMAINING MCC VALUES ==========")

# Any missing MCC should explicitly be marked UNKNOWN
merchants["mcc"] = merchants["mcc"].fillna("UNKNOWN")

# Make sure the validation flag is also complete
merchants["mcc_valid_flag"] = merchants["mcc_valid_flag"].fillna(False)

print("Remaining missing MCC:", merchants["mcc"].isna().sum())
print("UNKNOWN MCC count:", (merchants["mcc"] == "UNKNOWN").sum())

print("\nFinal MCC distribution:")
display(
    merchants["mcc"]
    .value_counts(dropna=False)
    .head(20)
)

========== FIXING REMAINING MCC VALUES ==========
Remaining missing MCC: 0
UNKNOWN MCC count: 693

Final MCC distribution:


mcc
UNKNOWN    693
4131       589
5311       580
5699       571
7011       557
4814       557
5912       550
5812       547
5411       531
5999       522
5942       513
Name: count, dtype: Int64

In [47]:
print("========== STANDARDIZING MERCHANT STATUS ==========")

# Clean raw status text
merchants["merchant_status"] = (
    merchants["merchant_status"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"[^A-Z0-9]+", "_", regex=True)
    .str.strip("_")
)

# Map fragmented statuses into canonical categories
status_map = {
    # Active variants
    "ACTIVE": "ACTIVE",
    "A": "ACTIVE",
    "ENABLED": "ACTIVE",
    "LIVE": "ACTIVE",

    # Inactive variants
    "INACTIVE": "INACTIVE",
    "I": "INACTIVE",
    "DISABLED": "INACTIVE",
    "CLOSED": "INACTIVE",
    "BLOCKED": "INACTIVE",

    # Suspended / restricted variants
    "SUSPENDED": "SUSPENDED",
    "SUSPEND": "SUSPENDED",
    "HOLD": "SUSPENDED",

    # Anything else
    "UNKNOWN": "UNKNOWN",
    "": "UNKNOWN"
}

merchants["merchant_status"] = (
    merchants["merchant_status"]
    .map(status_map)
    .fillna("UNKNOWN")
)

print("Merchant status standardization completed.")

print("\nStandardized merchant statuses:")

display(
    merchants["merchant_status"]
    .value_counts(dropna=False)
)

========== STANDARDIZING MERCHANT STATUS ==========
Merchant status standardization completed.

Standardized merchant statuses:


merchant_status
ACTIVE       5037
INACTIVE      807
SUSPENDED     296
UNKNOWN        70
Name: count, dtype: int64

In [48]:
print("========== CLEANING MERCHANT DATES & TICKET SIZE ==========")

# --------------------------------------------------
# 1. Clean onboarding date
# --------------------------------------------------

merchants["onboarding_date"] = pd.to_datetime(
    merchants["onboarding_date"],
    errors="coerce"
)

print("Missing onboarding dates:",
      merchants["onboarding_date"].isna().sum())

# --------------------------------------------------
# 2. Clean declared average ticket size
# --------------------------------------------------

merchants["declared_avg_ticket_size"] = (
    merchants["declared_avg_ticket_size"]
    .astype("string")
    .str.replace(r"[₹,\s]", "", regex=True)
)

merchants["declared_avg_ticket_size"] = pd.to_numeric(
    merchants["declared_avg_ticket_size"],
    errors="coerce"
)

# Negative ticket sizes are invalid
merchants.loc[
    merchants["declared_avg_ticket_size"] < 0,
    "declared_avg_ticket_size"
] = pd.NA

print(
    "Missing/invalid ticket sizes:",
    merchants["declared_avg_ticket_size"].isna().sum()
)

# --------------------------------------------------
# 3. Create data-quality flags
# --------------------------------------------------

merchants["onboarding_date_missing_flag"] = (
    merchants["onboarding_date"].isna()
)

merchants["ticket_size_missing_flag"] = (
    merchants["declared_avg_ticket_size"].isna()
)

# --------------------------------------------------
# 4. Display results
# --------------------------------------------------

print("\n========== MERCHANT DATE SUMMARY ==========")

print(
    "Earliest onboarding:",
    merchants["onboarding_date"].min()
)

print(
    "Latest onboarding:",
    merchants["onboarding_date"].max()
)

print("\n========== TICKET SIZE SUMMARY ==========")

display(
    merchants["declared_avg_ticket_size"].describe()
)

print("\n========== SAMPLE ==========")

display(
    merchants[
        [
            "merchant_id",
            "onboarding_date",
            "declared_avg_ticket_size",
            "onboarding_date_missing_flag",
            "ticket_size_missing_flag"
        ]
    ].head(10)
)

========== CLEANING MERCHANT DATES & TICKET SIZE ==========
Missing onboarding dates: 5328
Missing/invalid ticket sizes: 2497

========== MERCHANT DATE SUMMARY ==========
Earliest onboarding: 2023-01-01 00:00:00
Latest onboarding: 2026-02-28 00:00:00

========== TICKET SIZE SUMMARY ==========


count         3713.0
mean     1805.719227
std      1921.111069
min            54.42
25%           668.22
50%          1234.51
75%          2233.91
max         23536.65
Name: declared_avg_ticket_size, dtype: Float64


========== SAMPLE ==========


,merchant_id,onboarding_date,declared_avg_ticket_size,onboarding_date_missing_flag,ticket_size_missing_flag
0,MCH2849,2025-09-23,<NA>,False,True
1,MCH4314,NaT,986.05,True,False
2,MCH1986,NaT,<NA>,True,True
3,MCH3899,NaT,<NA>,True,True
4,MCH4859,NaT,<NA>,True,True
5,MCH6848,NaT,1272.49,True,False
6,MCH9933,NaT,<NA>,True,True
7,MCH2637,NaT,<NA>,True,True
8,MCH8708,NaT,2919.92,True,False
9,MCH2430,NaT,827.43,True,False


In [49]:
print("========== MERCHANT MASTER-DATA CONFLICT DETECTION ==========")

# Fields that should normally remain consistent for a merchant
conflict_fields = [
    "merchant_name",
    "mcc",
    "merchant_category",
    "business_type",
    "city",
    "state",
    "settlement_account",
    "merchant_status"
]

# --------------------------------------------------
# Build merchant-level conflict profile
# --------------------------------------------------

merchant_profile = (
    merchants
    .groupby("merchant_id")
    .agg(
        merchant_records=("merchant_id", "size"),

        distinct_names=("merchant_name", "nunique"),
        distinct_mcc=("mcc", "nunique"),
        distinct_categories=("merchant_category", "nunique"),
        distinct_business_types=("business_type", "nunique"),
        distinct_cities=("city", "nunique"),
        distinct_states=("state", "nunique"),
        distinct_settlement_accounts=("settlement_account", "nunique"),
        distinct_statuses=("merchant_status", "nunique")
    )
    .reset_index()
)

# --------------------------------------------------
# Conflict flags
# --------------------------------------------------

merchant_profile["name_conflict_flag"] = (
    merchant_profile["distinct_names"] > 1
)

merchant_profile["mcc_conflict_flag"] = (
    merchant_profile["distinct_mcc"] > 1
)

merchant_profile["category_conflict_flag"] = (
    merchant_profile["distinct_categories"] > 1
)

merchant_profile["business_type_conflict_flag"] = (
    merchant_profile["distinct_business_types"] > 1
)

merchant_profile["city_conflict_flag"] = (
    merchant_profile["distinct_cities"] > 1
)

merchant_profile["state_conflict_flag"] = (
    merchant_profile["distinct_states"] > 1
)

merchant_profile["settlement_account_conflict_flag"] = (
    merchant_profile["distinct_settlement_accounts"] > 1
)

merchant_profile["status_conflict_flag"] = (
    merchant_profile["distinct_statuses"] > 1
)

# --------------------------------------------------
# Overall conflict flag
# --------------------------------------------------

conflict_flags = [
    "name_conflict_flag",
    "mcc_conflict_flag",
    "category_conflict_flag",
    "business_type_conflict_flag",
    "city_conflict_flag",
    "state_conflict_flag",
    "settlement_account_conflict_flag",
    "status_conflict_flag"
]

merchant_profile["master_data_conflict_flag"] = (
    merchant_profile[conflict_flags].any(axis=1)
)

# --------------------------------------------------
# Conflict count
# --------------------------------------------------

merchant_profile["conflict_count"] = (
    merchant_profile[conflict_flags].sum(axis=1)
)

# --------------------------------------------------
# Summary
# --------------------------------------------------

print("Unique merchants:", len(merchant_profile))

print(
    "Merchants with master-data conflicts:",
    merchant_profile["master_data_conflict_flag"].sum()
)

print(
    "Merchants without conflicts:",
    (~merchant_profile["master_data_conflict_flag"]).sum()
)

print("\n========== CONFLICT TYPES ==========")

print(
    "Name conflicts:",
    merchant_profile["name_conflict_flag"].sum()
)

print(
    "MCC conflicts:",
    merchant_profile["mcc_conflict_flag"].sum()
)

print(
    "Category conflicts:",
    merchant_profile["category_conflict_flag"].sum()
)

print(
    "Business-type conflicts:",
    merchant_profile["business_type_conflict_flag"].sum()
)

print(
    "City conflicts:",
    merchant_profile["city_conflict_flag"].sum()
)

print(
    "State conflicts:",
    merchant_profile["state_conflict_flag"].sum()
)

print(
    "Settlement-account conflicts:",
    merchant_profile["settlement_account_conflict_flag"].sum()
)

print(
    "Status conflicts:",
    merchant_profile["status_conflict_flag"].sum()
)

# --------------------------------------------------
# Show suspicious merchants
# --------------------------------------------------

print("\n========== TOP MERCHANT MASTER-DATA CONFLICTS ==========")

display(
    merchant_profile[
        merchant_profile["master_data_conflict_flag"]
    ]
    .sort_values(
        ["conflict_count", "merchant_records"],
        ascending=False
    )
    .head(20)
)

========== MERCHANT MASTER-DATA CONFLICT DETECTION ==========
Unique merchants: 4480
Merchants with master-data conflicts: 1262
Merchants without conflicts: 3218

========== CONFLICT TYPES ==========
Name conflicts: 1211
MCC conflicts: 1130
Category conflicts: 1192
Business-type conflicts: 949
City conflicts: 1165
State conflicts: 1074
Settlement-account conflicts: 537
Status conflicts: 489

========== TOP MERCHANT MASTER-DATA CONFLICTS ==========


,merchant_id,merchant_records,distinct_names,distinct_mcc,distinct_categories,distinct_business_types,distinct_cities,distinct_states,distinct_settlement_accounts,distinct_statuses,name_conflict_flag,mcc_conflict_flag,category_conflict_flag,business_type_conflict_flag,city_conflict_flag,state_conflict_flag,settlement_account_conflict_flag,status_conflict_flag,master_data_conflict_flag,conflict_count
1545,MCH3659,6,5,5,5,3,5,4,3,3,True,True,True,True,True,True,True,True,True,8
2646,MCH6061,6,5,4,5,3,3,3,3,2,True,True,True,True,True,True,True,True,True,8
3483,MCH7912,6,5,5,5,3,4,4,2,2,True,True,True,True,True,True,True,True,True,8
3588,MCH8125,6,6,6,6,4,6,5,4,3,True,True,True,True,True,True,True,True,True,8
1626,MCH3837,5,4,4,4,4,3,2,3,2,True,True,True,True,True,True,True,True,True,8
2476,MCH5683,5,4,3,4,2,4,4,3,2,True,True,True,True,True,True,True,True,True,8
2953,MCH6743,5,4,4,4,2,4,4,4,2,True,True,True,True,True,True,True,True,True,8
3668,MCH8297,5,5,5,5,3,5,4,4,3,True,True,True,True,True,True,True,True,True,8
4191,MCH9393,5,5,5,5,3,4,3,5,2,True,True,True,True,True,True,True,True,True,8
595,MCH1625,4,3,3,3,3,3,3,2,2,True,True,True,True,True,True,True,True,True,8


In [50]:
print("========== BUILDING MERCHANT RISK PROFILE ==========")

# --------------------------------------------------
# Merchant risk indicators
# --------------------------------------------------

merchant_profile["missing_mcc_flag"] = (
    merchants.groupby("merchant_id")["mcc"]
    .apply(lambda x: x.isna().any() or (x == "UNKNOWN").any())
    .reindex(merchant_profile["merchant_id"])
    .fillna(False)
    .values
)

merchant_profile["missing_onboarding_date_flag"] = (
    merchants.groupby("merchant_id")["onboarding_date"]
    .apply(lambda x: x.isna().any())
    .reindex(merchant_profile["merchant_id"])
    .fillna(False)
    .values
)

merchant_profile["missing_ticket_size_flag"] = (
    merchants.groupby("merchant_id")["declared_avg_ticket_size"]
    .apply(lambda x: x.isna().any())
    .reindex(merchant_profile["merchant_id"])
    .fillna(False)
    .values
)

merchant_profile["missing_settlement_account_flag"] = (
    merchants.groupby("merchant_id")["settlement_account"]
    .apply(lambda x: x.isna().any())
    .reindex(merchant_profile["merchant_id"])
    .fillna(False)
    .values
)

# --------------------------------------------------
# Master-data risk score
# --------------------------------------------------

merchant_profile["master_data_risk_score"] = (
    merchant_profile["master_data_conflict_flag"].astype(int) * 4
    + merchant_profile["name_conflict_flag"].astype(int) * 2
    + merchant_profile["mcc_conflict_flag"].astype(int) * 2
    + merchant_profile["category_conflict_flag"].astype(int) * 2
    + merchant_profile["business_type_conflict_flag"].astype(int)
    + merchant_profile["city_conflict_flag"].astype(int)
    + merchant_profile["state_conflict_flag"].astype(int)
    + merchant_profile["settlement_account_conflict_flag"].astype(int) * 3
    + merchant_profile["status_conflict_flag"].astype(int) * 2
    + merchant_profile["missing_mcc_flag"].astype(int)
    + merchant_profile["missing_onboarding_date_flag"].astype(int)
    + merchant_profile["missing_ticket_size_flag"].astype(int)
    + merchant_profile["missing_settlement_account_flag"].astype(int)
)

# --------------------------------------------------
# Risk level
# --------------------------------------------------

def merchant_risk_level(score):
    if score >= 10:
        return "HIGH"
    elif score >= 5:
        return "MEDIUM"
    else:
        return "LOW"

merchant_profile["merchant_risk_level"] = (
    merchant_profile["master_data_risk_score"]
    .apply(merchant_risk_level)
)

# --------------------------------------------------
# Summary
# --------------------------------------------------

print("Merchant risk profile created successfully!")

print("\n========== MERCHANT RISK DISTRIBUTION ==========")

print(
    merchant_profile["merchant_risk_level"]
    .value_counts()
)

print("\n========== RISK SCORE SUMMARY ==========")

print(
    merchant_profile["master_data_risk_score"]
    .describe()
)

print("\n========== TOP HIGH-RISK MERCHANTS ==========")

display(
    merchant_profile[
        [
            "merchant_id",
            "merchant_records",
            "master_data_conflict_flag",
            "conflict_count",
            "name_conflict_flag",
            "mcc_conflict_flag",
            "category_conflict_flag",
            "settlement_account_conflict_flag",
            "status_conflict_flag",
            "master_data_risk_score",
            "merchant_risk_level"
        ]
    ]
    .sort_values(
        "master_data_risk_score",
        ascending=False
    )
    .head(20)
)

========== BUILDING MERCHANT RISK PROFILE ==========
Merchant risk profile created successfully!

========== MERCHANT RISK DISTRIBUTION ==========
merchant_risk_level
LOW       3218
HIGH      1212
MEDIUM      50
Name: count, dtype: int64

========== RISK SCORE SUMMARY ==========
count    4480.000000
mean        5.979018
std         6.922239
min         0.000000
25%         1.000000
50%         2.000000
75%        14.000000
max        22.000000
Name: master_data_risk_score, dtype: float64

========== TOP HIGH-RISK MERCHANTS ==========


,merchant_id,merchant_records,master_data_conflict_flag,conflict_count,name_conflict_flag,mcc_conflict_flag,category_conflict_flag,settlement_account_conflict_flag,status_conflict_flag,master_data_risk_score,merchant_risk_level
4291,MCH9609,3,True,8,True,True,True,True,True,22,HIGH
809,MCH2085,4,True,8,True,True,True,True,True,22,HIGH
3483,MCH7912,6,True,8,True,True,True,True,True,22,HIGH
333,MCH1060,3,True,8,True,True,True,True,True,22,HIGH
1547,MCH3662,4,True,8,True,True,True,True,True,22,HIGH
618,MCH1675,4,True,8,True,True,True,True,True,22,HIGH
1529,MCH3616,4,True,8,True,True,True,True,True,22,HIGH
595,MCH1625,4,True,8,True,True,True,True,True,22,HIGH
4005,MCH9004,3,True,8,True,True,True,True,True,22,HIGH
3820,MCH8615,3,True,8,True,True,True,True,True,22,HIGH


In [51]:
print("========== SAVING MERCHANT OUTPUTS ==========")

import os

output_dir = r"/mnt/data/upi_fraud_project"
os.makedirs(output_dir, exist_ok=True)

# --------------------------------------------------
# Save cleaned merchant dataset
# --------------------------------------------------

merchants.to_csv(
    os.path.join(output_dir, "merchants_clean.csv"),
    index=False
)

# --------------------------------------------------
# Save merchant risk / conflict profile
# --------------------------------------------------

merchant_profile.to_csv(
    os.path.join(output_dir, "merchant_profile.csv"),
    index=False
)

# --------------------------------------------------
# Verification
# --------------------------------------------------

print("Merchant outputs saved successfully!")

print("\nClean merchant dataset:")
print(os.path.join(output_dir, "merchants_clean.csv"))

print("\nMerchant risk profile:")
print(os.path.join(output_dir, "merchant_profile.csv"))

print("\nRows in merchants_clean:", len(merchants))
print("Rows in merchant_profile:", len(merchant_profile))

print("\nFiles exist:")

print(
    "merchants_clean.csv:",
    os.path.exists(
        os.path.join(output_dir, "merchants_clean.csv")
    )
)

print(
    "merchant_profile.csv:",
    os.path.exists(
        os.path.join(output_dir, "merchant_profile.csv")
    )
)

========== SAVING MERCHANT OUTPUTS ==========
Merchant outputs saved successfully!

Clean merchant dataset:
/mnt/data/upi_fraud_project\merchants_clean.csv

Merchant risk profile:
/mnt/data/upi_fraud_project\merchant_profile.csv

Rows in merchants_clean: 6210
Rows in merchant_profile: 4480

Files exist:
merchants_clean.csv: True
merchant_profile.csv: True


In [52]:
print("========== LOADING CHARGEBACK DATA ==========")

import json
import os
import pandas as pd

# Chargeback file location
chargeback_file = r"../data/raw/track1_chargebacks.json"

print("Checking file:")
print(os.path.abspath(chargeback_file))

# Load JSON
with open(chargeback_file, "r", encoding="utf-8") as f:
    chargeback_data = json.load(f)

print("\nChargeback JSON loaded successfully!")

# Check structure
print("\nJSON object type:")
print(type(chargeback_data))

if isinstance(chargeback_data, list):
    print("Number of chargeback records:", len(chargeback_data))
    
    chargebacks = pd.DataFrame(chargeback_data)

elif isinstance(chargeback_data, dict):
    print("JSON contains dictionary keys:")
    print(chargeback_data.keys())
    
    # Try common record containers
    if "data" in chargeback_data:
        chargebacks = pd.DataFrame(chargeback_data["data"])
    elif "chargebacks" in chargeback_data:
        chargebacks = pd.DataFrame(chargeback_data["chargebacks"])
    else:
        chargebacks = pd.DataFrame([chargeback_data])

else:
    raise ValueError("Unexpected JSON structure.")

print("\nChargeback dataset created successfully!")
print("Rows:", len(chargebacks))
print("Columns:", len(chargebacks.columns))

print("\nChargeback columns:")
print(chargebacks.columns.tolist())

print("\nFirst 5 records:")
display(chargebacks.head())

========== LOADING CHARGEBACK DATA ==========
Checking file:
C:\Users\VIKASH\Desktop\UPI-Fraud-Ring-Merchant-Analytics\data\raw\track1_chargebacks.json

Chargeback JSON loaded successfully!

JSON object type:
<class 'list'>
Number of chargeback records: 2884

Chargeback dataset created successfully!
Rows: 2884
Columns: 13

Chargeback columns:
['complaint_id', 'txn_id', 'user_id', 'merchant_id', 'transaction_timestamp', 'reported_timestamp', 'disputed_amount', 'reason_code', 'complaint_text', 'resolution_status', 'bank_response_timestamp', 'severity', 'channel']

First 5 records:


,complaint_id,txn_id,user_id,merchant_id,transaction_timestamp,reported_timestamp,disputed_amount,reason_code,complaint_text,resolution_status,bank_response_timestamp,severity,channel
0,CBK0002082,TXN00004325,usr97580,mch1127,2026/01/28,02-01-2026,,Merchant Not Delivered,Customer says amount was debited twice.,CLOSED,2026-02-10 03:19:10,Critical,ivr
1,CBK0001941,TXN00003720,USR54113,3835,25/02/2026 10:24 AM,1772691855,414.69,login compromised,User reports money deducted but merchant denie...,In Progress,12/03/2026 06:24 AM,H,Call Center
2,CBK0001799,TXN00012539,USR17980,mch3700,2026/02/04,02-05-2026,"Rs. 7,039",customer issue,merchant service was not delivered after payment.,OPEN,06/03/2026,P4,IVR
3,CBK0002465,TXN00017802,USR76148,MCH4534,30-Mar-2026,01-Apr-2026,1303.05,no service,Suspicious high-value payment disputed by cust...,Rejected,07-Apr-2026,H,ivr
4,CBK0001870,TXN00015944,USR24660,MCH1686,09-Jan-2026,1768424501,1459.42,merchant service issue,Merchant service was not delivered after payment.,In Progress,01-26-2026,Low,App


In [53]:
print("========== CHARGEBACK DATA QUALITY CHECK ==========")

print("Rows:", len(chargebacks))
print("Columns:", len(chargebacks.columns))

# -----------------------------
# MISSING VALUES
# -----------------------------
print("\n========== MISSING VALUES ==========")

missing_cb = chargebacks.isna().sum()
missing_cb = missing_cb[missing_cb > 0].sort_values(ascending=False)

if len(missing_cb) > 0:
    print(missing_cb)
else:
    print("No missing values found.")


# -----------------------------
# DUPLICATE RECORDS
# -----------------------------
print("\n========== DUPLICATE ROWS ==========")

duplicate_cb = chargebacks.duplicated().sum()

print("Exact duplicate rows:", duplicate_cb)


# -----------------------------
# COMPLAINT ID CHECK
# -----------------------------
print("\n========== COMPLAINT ID CHECK ==========")

print("Unique complaint IDs:", chargebacks["complaint_id"].nunique())
print("Missing complaint IDs:", chargebacks["complaint_id"].isna().sum())


# -----------------------------
# TRANSACTION ID CHECK
# -----------------------------
print("\n========== TRANSACTION ID CHECK ==========")

print("Unique transaction IDs:", chargebacks["txn_id"].nunique())
print("Missing transaction IDs:", chargebacks["txn_id"].isna().sum())


# -----------------------------
# USER ID CHECK
# -----------------------------
print("\n========== USER ID CHECK ==========")

print("Unique user IDs:", chargebacks["user_id"].nunique())
print("Missing user IDs:", chargebacks["user_id"].isna().sum())


# -----------------------------
# MERCHANT ID CHECK
# -----------------------------
print("\n========== MERCHANT ID CHECK ==========")

print("Unique merchant IDs:", chargebacks["merchant_id"].nunique())
print("Missing merchant IDs:", chargebacks["merchant_id"].isna().sum())


# -----------------------------
# REASON CODE
# -----------------------------
print("\n========== REASON CODE ==========")

display(
    chargebacks["reason_code"]
    .value_counts(dropna=False)
)


# -----------------------------
# RESOLUTION STATUS
# -----------------------------
print("\n========== RESOLUTION STATUS ==========")

display(
    chargebacks["resolution_status"]
    .value_counts(dropna=False)
)


# -----------------------------
# SEVERITY
# -----------------------------
print("\n========== SEVERITY ==========")

display(
    chargebacks["severity"]
    .value_counts(dropna=False)
)


# -----------------------------
# CHANNEL
# -----------------------------
print("\n========== CHANNEL ==========")

display(
    chargebacks["channel"]
    .value_counts(dropna=False)
)


# -----------------------------
# DISPUTED AMOUNT
# -----------------------------
print("\n========== DISPUTED AMOUNT ==========")

print("Data type:", chargebacks["disputed_amount"].dtype)

print("\nSample disputed amounts:")
display(chargebacks["disputed_amount"].head(20))


print("\n========== QUALITY CHECK COMPLETE ==========")

========== CHARGEBACK DATA QUALITY CHECK ==========
Rows: 2884
Columns: 13

========== MISSING VALUES ==========
No missing values found.

========== DUPLICATE ROWS ==========
Exact duplicate rows: 84

========== COMPLAINT ID CHECK ==========
Unique complaint IDs: 2800
Missing complaint IDs: 0

========== TRANSACTION ID CHECK ==========
Unique transaction IDs: 2582
Missing transaction IDs: 0

========== USER ID CHECK ==========
Unique user IDs: 2454
Missing user IDs: 0

========== MERCHANT ID CHECK ==========
Unique merchant IDs: 2051
Missing merchant IDs: 0

========== REASON CODE ==========


reason_code
customer issue              109
delivery issue              106
item not received           106
dispute raised              103
charged twice               102
no service                   97
account hacked               94
Merchant Not Delivered       93
extra amount deducted        93
Duplicate Debit              92
DUP_DEBIT                    90
amount mismatch              90
Customer Dispute             89
login compromised            88
ATO                          88
merchant service issue       86
not delivered                86
Account Takeover             85
complaint                    84
not done by me               84
service failed               82
Wrong Amount                 82
double debit                 80
Unauthorized Transaction     80
unauth txn                   80
unauthorized_transaction     74
incorrect amount             74
FRAUD                        74
Fraud Suspected              70
Service Not Provided         69
UNAUTHORISED                


========== RESOLUTION STATUS ==========


resolution_status
Pending Bank    247
Rejected        246
CLOSED          236
Open            233
RESOLVED        227
PENDING_BANK    225
Closed          224
REJECTED        216
IN_PROGRESS     214
OPEN            207
Resolved        207
WIP             206
In Progress     196
Name: count, dtype: int64


========== SEVERITY ==========


...


========== CHANNEL ==========


channel
Email          382
Branch         376
ivr            367
CHATBOT        367
IVR            362
chatbot        358
App            355
Call Center    317
Name: count, dtype: int64


========== DISPUTED AMOUNT ==========
Data type: object

Sample disputed amounts:


0               
1         414.69
2      Rs. 7,039
3        1303.05
4        1459.42
5        Rs. 548
6      ₹1,949.60
7        4193.52
8         930.83
9               
10       1805.23
11      4,853.70
12    Rs. 12,334
13      7,638.70
14              
15        174.11
16       Rs. 305
17     Rs. 4,952
18       1126.18
19        513.84
Name: disputed_amount, dtype: object


========== QUALITY CHECK COMPLETE ==========


In [54]:
print("========== STANDARDIZING CHARGEBACK DATA ==========")

# -------------------------------------------------
# 1. STANDARDIZE IDs
# -------------------------------------------------

def normalize_user_id(x):
    if pd.isna(x):
        return pd.NA

    s = str(x).strip().upper()

    # Remove spaces and common separators
    s = s.replace(" ", "").replace("-", "").replace("_", "")

    # Keep only digits if the value starts with USR
    if s.startswith("USR"):
        digits = "".join(ch for ch in s[3:] if ch.isdigit())
        if digits:
            return "USR" + digits

    # If only digits are present
    digits = "".join(ch for ch in s if ch.isdigit())

    if digits:
        return "USR" + digits

    return s


def normalize_merchant_id(x):
    if pd.isna(x):
        return pd.NA

    s = str(x).strip().upper()

    # Remove spaces and common separators
    s = s.replace(" ", "").replace("-", "").replace("_", "")

    if s.startswith("MCH"):
        digits = "".join(ch for ch in s[3:] if ch.isdigit())
        if digits:
            return "MCH" + digits

    digits = "".join(ch for ch in s if ch.isdigit())

    if digits:
        return "MCH" + digits

    return s


def normalize_txn_id(x):
    if pd.isna(x):
        return pd.NA

    s = str(x).strip().upper()

    s = s.replace(" ", "").replace("-", "").replace("_", "")

    if s.startswith("TXN"):
        digits = "".join(ch for ch in s[3:] if ch.isdigit())
        if digits:
            return "TXN" + digits

    digits = "".join(ch for ch in s if ch.isdigit())

    if digits:
        return "TXN" + digits

    return s


def normalize_complaint_id(x):
    if pd.isna(x):
        return pd.NA

    s = str(x).strip().upper()

    s = s.replace(" ", "").replace("-", "").replace("_", "")

    if s.startswith("CBK"):
        digits = "".join(ch for ch in s[3:] if ch.isdigit())
        if digits:
            return "CBK" + digits

    digits = "".join(ch for ch in s if ch.isdigit())

    if digits:
        return "CBK" + digits

    return s


chargebacks["complaint_id"] = (
    chargebacks["complaint_id"]
    .apply(normalize_complaint_id)
)

chargebacks["txn_id"] = (
    chargebacks["txn_id"]
    .apply(normalize_txn_id)
)

chargebacks["user_id"] = (
    chargebacks["user_id"]
    .apply(normalize_user_id)
)

chargebacks["merchant_id"] = (
    chargebacks["merchant_id"]
    .apply(normalize_merchant_id)
)


# -------------------------------------------------
# 2. STANDARDIZE TEXT FIELDS
# -------------------------------------------------

text_columns = [
    "complaint_text",
    "reason_code",
    "resolution_status",
    "severity",
    "channel"
]

for col in text_columns:
    if col in chargebacks.columns:
        chargebacks[col] = (
            chargebacks[col]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )


# -------------------------------------------------
# 3. STANDARDIZE REASON CODE
# -------------------------------------------------

def normalize_reason(x):
    if pd.isna(x):
        return "UNKNOWN"

    s = str(x).strip().upper()

    s = s.replace("-", " ")
    s = s.replace("_", " ")
    s = " ".join(s.split())

    # Fraud-related reasons
    if (
        "FRAUD" in s
        or "UNAUTHORIZED" in s
        or "UNAUTH" in s
        or "ACCOUNT HACK" in s
        or "ACCOUNT TAKEOVER" in s
        or "ATO" == s
        or "LOGIN COMPROMISED" in s
        or "SCAM" in s
        or "SUSPICIOUS TRANSACTION" in s
    ):
        return "FRAUD_OR_UNAUTHORIZED"

    # Duplicate debit/payment
    if (
        "DUPLICATE" in s
        or "DOUBLE DEBIT" in s
        or "CHARGED TWICE" in s
    ):
        return "DUPLICATE_PAYMENT"

    # Amount-related
    if (
        "AMOUNT" in s
        or "WRONG AMOUNT" in s
        or "EXTRA AMOUNT" in s
    ):
        return "AMOUNT_ISSUE"

    # Delivery/service issues
    if (
        "NOT DELIVERED" in s
        or "DELIVERY" in s
        or "ITEM NOT RECEIVED" in s
        or "SERVICE NOT" in s
        or "NO SERVICE" in s
        or "SERVICE FAILED" in s
        or "SERVICE ISSUE" in s
    ):
        return "SERVICE_OR_DELIVERY"

    # Customer disputes
    if (
        "CUSTOMER" in s
        or "COMPLAINT" in s
        or "DISPUTE" in s
    ):
        return "CUSTOMER_DISPUTE"

    return s


chargebacks["reason_group"] = (
    chargebacks["reason_code"]
    .apply(normalize_reason)
)


# -------------------------------------------------
# 4. STANDARDIZE RESOLUTION STATUS
# -------------------------------------------------

def normalize_resolution(x):
    if pd.isna(x):
        return "UNKNOWN"

    s = str(x).strip().upper()

    s = s.replace("-", "_")
    s = " ".join(s.split())

    if "RESOLVED" in s:
        return "RESOLVED"

    if "REJECT" in s:
        return "REJECTED"

    if "CLOSED" in s:
        return "CLOSED"

    if "PENDING" in s:
        return "PENDING"

    if "PROGRESS" in s or s == "WIP":
        return "IN_PROGRESS"

    if "OPEN" in s:
        return "OPEN"

    return "UNKNOWN"


chargebacks["resolution_status"] = (
    chargebacks["resolution_status"]
    .apply(normalize_resolution)
)


# -------------------------------------------------
# 5. STANDARDIZE SEVERITY
# -------------------------------------------------

def normalize_severity(x):
    if pd.isna(x):
        return "UNKNOWN"

    s = str(x).strip().upper()

    if s in ["P1", "HIGH", "H"]:
        return "HIGH"

    if s in ["P2", "P3", "MEDIUM", "M"]:
        return "MEDIUM"

    if s in ["P4", "LOW", "L"]:
        return "LOW"

    return "UNKNOWN"


chargebacks["severity"] = (
    chargebacks["severity"]
    .apply(normalize_severity)
)


# -------------------------------------------------
# 6. STANDARDIZE CHANNEL
# -------------------------------------------------

chargebacks["channel"] = (
    chargebacks["channel"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace("-", "_")
    .str.replace(" ", "_")
)


# -------------------------------------------------
# 7. SHOW RESULTS
# -------------------------------------------------

print("Chargeback ID standardization completed.")

print("\n========== STANDARDIZED REASON GROUPS ==========")
display(
    chargebacks["reason_group"]
    .value_counts(dropna=False)
)

print("\n========== STANDARDIZED RESOLUTION STATUS ==========")
display(
    chargebacks["resolution_status"]
    .value_counts(dropna=False)
)

print("\n========== STANDARDIZED SEVERITY ==========")
display(
    chargebacks["severity"]
    .value_counts(dropna=False)
)

print("\n========== STANDARDIZED CHANNEL ==========")
display(
    chargebacks["channel"]
    .value_counts(dropna=False)
)

print("\n========== STANDARDIZED ID SAMPLE ==========")
display(
    chargebacks[
        ["complaint_id", "txn_id", "user_id", "merchant_id"]
    ].head(10)
)

print("\n========== STEP 21 COMPLETE ==========")

========== STANDARDIZING CHARGEBACK DATA ==========
Chargeback ID standardization completed.

========== STANDARDIZED REASON GROUPS ==========


reason_group
FRAUD_OR_UNAUTHORIZED    987
SERVICE_OR_DELIVERY      725
CUSTOMER_DISPUTE         385
AMOUNT_ISSUE             339
DUPLICATE_PAYMENT        274
DUP DEBIT                 90
NOT DONE BY ME            84
Name: count, dtype: int64


========== STANDARDIZED RESOLUTION STATUS ==========


resolution_status
IN_PROGRESS    616
PENDING        472
REJECTED       462
CLOSED         460
OPEN           440
RESOLVED       434
Name: count, dtype: int64


========== STANDARDIZED SEVERITY ==========


...


========== STANDARDIZED CHANNEL ==========


channel
IVR            729
CHATBOT        725
EMAIL          382
BRANCH         376
APP            355
CALL_CENTER    317
Name: count, dtype: Int64


========== STANDARDIZED ID SAMPLE ==========


,complaint_id,txn_id,user_id,merchant_id
0,CBK0002082,TXN00004325,USR97580,MCH1127
1,CBK0001941,TXN00003720,USR54113,MCH3835
2,CBK0001799,TXN00012539,USR17980,MCH3700
3,CBK0002465,TXN00017802,USR76148,MCH4534
4,CBK0001870,TXN00015944,USR24660,MCH1686
5,CBK0002663,TXN00009741,USR40631,MCH9584
6,CBK0000782,TXN00015086,USR58789,MCH6008
7,CBK0001838,TXN00014426,USR30157,MCH4526
8,CBK0001095,TXN00015316,USR86750,MCH5980
9,CBK0002541,TXN00002822,USR11987,MCH3992



========== STEP 21 COMPLETE ==========


In [57]:
print("========== CLEANING CHARGEBACK AMOUNTS & TIMESTAMPS ==========")

import pandas as pd
import numpy as np
import re


# ---------------------------------------------------------
# 1. CLEAN DISPUTED AMOUNT
# ---------------------------------------------------------

def clean_disputed_amount(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if value == "":
        return np.nan

    value = re.sub(r'₹|Rs\.?|INR', '', value, flags=re.IGNORECASE)
    value = value.replace(',', '').strip()
    value = re.sub(r'[^0-9.]', '', value)

    if value == "":
        return np.nan

    try:
        return float(value)
    except:
        return np.nan


chargebacks["disputed_amount_clean"] = (
    chargebacks["disputed_amount"]
    .apply(clean_disputed_amount)
)

chargebacks["disputed_amount_missing_flag"] = (
    chargebacks["disputed_amount_clean"].isna()
)

print("\nDisputed amount cleaning completed.")

print("\n========== DISPUTED AMOUNT SUMMARY ==========")

print(
    chargebacks["disputed_amount_clean"].describe()
)

print(
    "\nMissing/invalid disputed amounts:",
    chargebacks["disputed_amount_missing_flag"].sum()
)


print("========== FIXING MIXED TIMESTAMP PARSING ==========")

import pandas as pd
import numpy as np


def parse_mixed_timestamp_robust(series):

    result = pd.Series(
        pd.NaT,
        index=series.index,
        dtype="datetime64[ns]"
    )

    values = series.astype("string").str.strip()

    # -----------------------------------------------------
    # 1. Identify numeric epoch timestamps
    # -----------------------------------------------------

    numeric = pd.to_numeric(values, errors="coerce")

    numeric_mask = numeric.notna()

    if numeric_mask.any():

        # Epoch seconds
        result.loc[numeric_mask] = pd.to_datetime(
            numeric.loc[numeric_mask],
            unit="s",
            errors="coerce"
        )

    # -----------------------------------------------------
    # 2. Parse remaining mixed date formats
    # -----------------------------------------------------

    remaining = result.isna()

    if remaining.any():

        try:
            # Pandas versions supporting format="mixed"
            result.loc[remaining] = pd.to_datetime(
                values.loc[remaining],
                format="mixed",
                errors="coerce",
                dayfirst=False
            )

        except:

            # Fallback for older Pandas versions
            result.loc[remaining] = pd.to_datetime(
                values.loc[remaining],
                errors="coerce",
                dayfirst=False
            )

    # -----------------------------------------------------
    # 3. Try day-first parsing for unresolved values
    # -----------------------------------------------------

    remaining = result.isna()

    if remaining.any():

        try:

            result.loc[remaining] = pd.to_datetime(
                values.loc[remaining],
                format="mixed",
                errors="coerce",
                dayfirst=True
            )

        except:

            result.loc[remaining] = pd.to_datetime(
                values.loc[remaining],
                errors="coerce",
                dayfirst=True
            )

    return result


# ---------------------------------------------------------
# PARSE ALL THREE TIMESTAMP COLUMNS
# ---------------------------------------------------------

chargebacks["transaction_timestamp_clean"] = (
    parse_mixed_timestamp_robust(
        chargebacks["transaction_timestamp"]
    )
)

chargebacks["report_timestamp_clean"] = (
    parse_mixed_timestamp_robust(
        chargebacks["reported_timestamp"]
    )
)

chargebacks["bank_response_timestamp_clean"] = (
    parse_mixed_timestamp_robust(
        chargebacks["bank_response_timestamp"]
    )
)


# ---------------------------------------------------------
# FLAGS
# ---------------------------------------------------------

chargebacks["transaction_timestamp_missing_flag"] = (
    chargebacks["transaction_timestamp_clean"].isna()
)

chargebacks["report_timestamp_missing_flag"] = (
    chargebacks["report_timestamp_clean"].isna()
)

chargebacks["bank_response_timestamp_missing_flag"] = (
    chargebacks["bank_response_timestamp_clean"].isna()
)


# ---------------------------------------------------------
# REPORTING DELAY
# ---------------------------------------------------------

chargebacks["reporting_delay_hours"] = (
    chargebacks["report_timestamp_clean"]
    - chargebacks["transaction_timestamp_clean"]
).dt.total_seconds() / 3600


chargebacks["negative_reporting_delay_flag"] = (
    chargebacks["reporting_delay_hours"] < 0
)


# ---------------------------------------------------------
# BANK RESPONSE DELAY
# ---------------------------------------------------------

chargebacks["bank_response_delay_hours"] = (
    chargebacks["bank_response_timestamp_clean"]
    - chargebacks["report_timestamp_clean"]
).dt.total_seconds() / 3600


chargebacks["negative_bank_response_delay_flag"] = (
    chargebacks["bank_response_delay_hours"] < 0
)


# ---------------------------------------------------------
# RESULTS
# ---------------------------------------------------------

print("\n========== TIMESTAMP RESULTS ==========")

print(
    "Missing transaction timestamps:",
    chargebacks["transaction_timestamp_missing_flag"].sum()
)

print(
    "Missing report timestamps:",
    chargebacks["report_timestamp_missing_flag"].sum()
)

print(
    "Missing bank response timestamps:",
    chargebacks["bank_response_timestamp_missing_flag"].sum()
)

print(
    "Negative reporting delays:",
    chargebacks["negative_reporting_delay_flag"].sum()
)

print(
    "Negative bank response delays:",
    chargebacks["negative_bank_response_delay_flag"].sum()
)


print("\n========== REPORTING DELAY ==========")

print(
    chargebacks["reporting_delay_hours"].describe()
)


# ---------------------------------------------------------
# SAMPLE
# ---------------------------------------------------------

print("\n========== TIMESTAMP SAMPLE ==========")

display(
    chargebacks[
        [
            "transaction_timestamp",
            "transaction_timestamp_clean",
            "reported_timestamp",
            "report_timestamp_clean",
            "bank_response_timestamp",
            "bank_response_timestamp_clean",
            "reporting_delay_hours"
        ]
    ].head(15)
)


print("\n========== TIMESTAMP FIX COMPLETE ==========")

# ---------------------------------------------------------
# 3. TIMESTAMP QUALITY FLAGS
# ---------------------------------------------------------

chargebacks["transaction_timestamp_missing_flag"] = (
    chargebacks["transaction_timestamp_clean"].isna()
)

chargebacks["report_timestamp_missing_flag"] = (
    chargebacks["report_timestamp_clean"].isna()
)

chargebacks["bank_response_timestamp_missing_flag"] = (
    chargebacks["bank_response_timestamp_clean"].isna()
)


# ---------------------------------------------------------
# 4. REPORTING DELAY
# ---------------------------------------------------------

chargebacks["reporting_delay_hours"] = (
    chargebacks["report_timestamp_clean"]
    - chargebacks["transaction_timestamp_clean"]
).dt.total_seconds() / 3600


chargebacks["negative_reporting_delay_flag"] = (
    chargebacks["reporting_delay_hours"] < 0
)


# ---------------------------------------------------------
# 5. BANK RESPONSE DELAY
# ---------------------------------------------------------

chargebacks["bank_response_delay_hours"] = (
    chargebacks["bank_response_timestamp_clean"]
    - chargebacks["report_timestamp_clean"]
).dt.total_seconds() / 3600


chargebacks["negative_bank_response_delay_flag"] = (
    chargebacks["bank_response_delay_hours"] < 0
)


# ---------------------------------------------------------
# 6. SUMMARY
# ---------------------------------------------------------

print("\n========== TIMESTAMP SUMMARY ==========")

print(
    "Missing transaction timestamps:",
    chargebacks["transaction_timestamp_missing_flag"].sum()
)

print(
    "Missing report timestamps:",
    chargebacks["report_timestamp_missing_flag"].sum()
)

print(
    "Missing bank response timestamps:",
    chargebacks["bank_response_timestamp_missing_flag"].sum()
)

print(
    "Negative reporting delays:",
    chargebacks["negative_reporting_delay_flag"].sum()
)

print(
    "Negative bank response delays:",
    chargebacks["negative_bank_response_delay_flag"].sum()
)


print("\n========== REPORTING DELAY SUMMARY ==========")

print(
    chargebacks["reporting_delay_hours"].describe()
)


# ---------------------------------------------------------
# 7. SAMPLE
# ---------------------------------------------------------

print("\n========== CLEANED CHARGEBACK SAMPLE ==========")

display(
    chargebacks[
        [
            "complaint_id",
            "txn_id",
            "user_id",
            "merchant_id",
            "disputed_amount",
            "disputed_amount_clean",
            "transaction_timestamp_clean",
            "report_timestamp_clean",
            "reporting_delay_hours"
        ]
    ].head(10)
)


print("\n========== STEP 22 COMPLETE ==========")

========== CLEANING CHARGEBACK AMOUNTS & TIMESTAMPS ==========

Disputed amount cleaning completed.

========== DISPUTED AMOUNT SUMMARY ==========
count     2701.000000
mean      3045.486109
std       3813.384614
min         50.000000
25%        913.380000
50%       1819.680000
75%       3670.720000
max      45384.610000
Name: disputed_amount_clean, dtype: float64

Missing/invalid disputed amounts: 183
========== FIXING MIXED TIMESTAMP PARSING ==========

========== TIMESTAMP RESULTS ==========
Missing transaction timestamps: 235
Missing report timestamps: 209
Missing bank response timestamps: 718
Negative reporting delays: 351
Negative bank response delays: 246

========== REPORTING DELAY ==========
count    2461.000000
mean       83.542518
std      1885.873525
min     -7740.382500
25%        24.000000
50%        72.000000
75%       240.000000
max      7861.750000
Name: reporting_delay_hours, dtype: float64

========== TIMESTAMP SAMPLE ==========


,transaction_timestamp,transaction_timestamp_clean,reported_timestamp,report_timestamp_clean,bank_response_timestamp,bank_response_timestamp_clean,reporting_delay_hours
0,2026/01/28,2026-01-28 00:00:00,02-01-2026,2026-02-01 00:00:00,2026-02-10 03:19:10,2026-02-10 03:19:10,96.000000
1,25/02/2026 10:24 AM,2026-02-25 10:24:00,1772691855,2026-03-05 06:24:15,12/03/2026 06:24 AM,2026-12-03 06:24:00,188.004167
2,2026/02/04,2026-02-04 00:00:00,02-05-2026,2026-02-05 00:00:00,06/03/2026,2026-06-03 00:00:00,24.000000
3,30-Mar-2026,2026-03-30 00:00:00,01-Apr-2026,2026-04-01 00:00:00,07-Apr-2026,2026-04-07 00:00:00,48.000000
4,09-Jan-2026,2026-01-09 00:00:00,1768424501,2026-01-14 21:01:41,01-26-2026,2026-01-26 00:00:00,141.028056
5,02-05-2026,2026-02-05 00:00:00,02-06-2026,2026-02-06 00:00:00,02-23-2026,2026-02-23 00:00:00,24.000000
6,05/02/2026,2026-05-02 00:00:00,02-10-2026,2026-02-10 00:00:00,03-Mar-2026,2026-03-03 00:00:00,-1944.000000
7,2026-03-26 10:17:08,2026-03-26 10:17:08,2026-04-06 04:17:08,2026-04-06 04:17:08,18-Apr-2026,2026-04-18 00:00:00,258.000000
8,2026/01/12,2026-01-12 00:00:00,13/01/2026,2026-01-13 00:00:00,14-Jan-2026,2026-01-14 00:00:00,24.000000
9,2026-03-18 10:04:06,2026-03-18 10:04:06,23/03/2026,2026-03-23 00:00:00,30/03/2026,2026-03-30 00:00:00,109.931667



========== TIMESTAMP FIX COMPLETE ==========

========== TIMESTAMP SUMMARY ==========
Missing transaction timestamps: 235
Missing report timestamps: 209
Missing bank response timestamps: 718
Negative reporting delays: 351
Negative bank response delays: 246

========== REPORTING DELAY SUMMARY ==========
count    2461.000000
mean       83.542518
std      1885.873525
min     -7740.382500
25%        24.000000
50%        72.000000
75%       240.000000
max      7861.750000
Name: reporting_delay_hours, dtype: float64

========== CLEANED CHARGEBACK SAMPLE ==========


,complaint_id,txn_id,user_id,merchant_id,disputed_amount,disputed_amount_clean,transaction_timestamp_clean,report_timestamp_clean,reporting_delay_hours
0,CBK0002082,TXN00004325,USR97580,MCH1127,,NaN,2026-01-28 00:00:00,2026-02-01 00:00:00,96.000000
1,CBK0001941,TXN00003720,USR54113,MCH3835,414.69,414.69,2026-02-25 10:24:00,2026-03-05 06:24:15,188.004167
2,CBK0001799,TXN00012539,USR17980,MCH3700,"Rs. 7,039",7039.00,2026-02-04 00:00:00,2026-02-05 00:00:00,24.000000
3,CBK0002465,TXN00017802,USR76148,MCH4534,1303.05,1303.05,2026-03-30 00:00:00,2026-04-01 00:00:00,48.000000
4,CBK0001870,TXN00015944,USR24660,MCH1686,1459.42,1459.42,2026-01-09 00:00:00,2026-01-14 21:01:41,141.028056
5,CBK0002663,TXN00009741,USR40631,MCH9584,Rs. 548,548.00,2026-02-05 00:00:00,2026-02-06 00:00:00,24.000000
6,CBK0000782,TXN00015086,USR58789,MCH6008,"₹1,949.60",1949.60,2026-05-02 00:00:00,2026-02-10 00:00:00,-1944.000000
7,CBK0001838,TXN00014426,USR30157,MCH4526,4193.52,4193.52,2026-03-26 10:17:08,2026-04-06 04:17:08,258.000000
8,CBK0001095,TXN00015316,USR86750,MCH5980,930.83,930.83,2026-01-12 00:00:00,2026-01-13 00:00:00,24.000000
9,CBK0002541,TXN00002822,USR11987,MCH3992,,NaN,2026-03-18 10:04:06,2026-03-23 00:00:00,109.931667



========== STEP 22 COMPLETE ==========


In [58]:
print("========== STEP 22: CLEANING CHARGEBACK AMOUNTS & TIMESTAMPS ==========")

import pandas as pd
import numpy as np
import re


# =========================================================
# 1. CLEAN DISPUTED AMOUNT
# =========================================================

def clean_disputed_amount(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    # Blank values
    if value == "":
        return np.nan

    # Remove currency symbols/text
    value = re.sub(
        r"₹|Rs\.?|INR",
        "",
        value,
        flags=re.IGNORECASE
    )

    # Remove commas
    value = value.replace(",", "")

    # Remove any remaining unwanted characters
    value = re.sub(
        r"[^0-9.]",
        "",
        value
    )

    # If nothing remains
    if value == "":
        return np.nan

    try:
        return float(value)
    except:
        return np.nan


chargebacks["disputed_amount_clean"] = (
    chargebacks["disputed_amount"]
    .apply(clean_disputed_amount)
)


# Missing amount flag
chargebacks["disputed_amount_missing_flag"] = (
    chargebacks["disputed_amount_clean"].isna()
)


print("\nDisputed amount cleaning completed.")

print("\n========== DISPUTED AMOUNT SUMMARY ==========")

display(
    chargebacks["disputed_amount_clean"].describe()
)

print(
    "Missing/invalid disputed amounts:",
    chargebacks["disputed_amount_missing_flag"].sum()
)


# =========================================================
# 2. ROBUST MIXED TIMESTAMP PARSER
# =========================================================

def parse_timestamp_correct(value):

    if pd.isna(value):
        return pd.NaT

    s = str(value).strip()

    if s == "":
        return pd.NaT


    # -----------------------------------------------------
    # A. Unix epoch timestamp
    # Example:
    # 1772691855
    # -----------------------------------------------------

    if re.fullmatch(r"\d{10}", s):

        return pd.to_datetime(
            int(s),
            unit="s",
            errors="coerce"
        )


    # -----------------------------------------------------
    # B. YYYY/MM/DD or YYYY-MM-DD
    # Examples:
    # 2026/01/28
    # 2026-03-26
    # 2026-03-26 10:17:08
    # -----------------------------------------------------

    if re.match(
        r"^\d{4}[/-]\d{1,2}[/-]\d{1,2}",
        s
    ):

        return pd.to_datetime(
            s,
            format="mixed",
            errors="coerce"
        )


    # -----------------------------------------------------
    # C. DD/MM/YYYY
    # Examples:
    # 05/02/2026
    # 25/02/2026 10:24 AM
    # 13/01/2026
    # -----------------------------------------------------

    if re.match(
        r"^\d{1,2}/\d{1,2}/\d{4}",
        s
    ):

        return pd.to_datetime(
            s,
            format="mixed",
            dayfirst=True,
            errors="coerce"
        )


    # -----------------------------------------------------
    # D. MM-DD-YYYY
    # Examples:
    # 02-10-2026
    # 01-26-2026
    # 02-17-2026
    # -----------------------------------------------------

    if re.match(
        r"^\d{1,2}-\d{1,2}-\d{4}",
        s
    ):

        return pd.to_datetime(
            s,
            format="mixed",
            dayfirst=False,
            errors="coerce"
        )


    # -----------------------------------------------------
    # E. Dates containing month names
    # Examples:
    # 30-Mar-2026
    # 07-Apr-2026
    # 18-Apr-2026
    # -----------------------------------------------------

    if re.search(
        r"[A-Za-z]{3,9}",
        s
    ):

        return pd.to_datetime(
            s,
            format="mixed",
            errors="coerce"
        )


    # -----------------------------------------------------
    # F. Final fallback
    # -----------------------------------------------------

    return pd.to_datetime(
        s,
        format="mixed",
        errors="coerce"
    )


# =========================================================
# 3. APPLY TIMESTAMP CLEANING
# =========================================================

chargebacks["transaction_timestamp_clean"] = (
    chargebacks["transaction_timestamp"]
    .apply(parse_timestamp_correct)
)

chargebacks["report_timestamp_clean"] = (
    chargebacks["reported_timestamp"]
    .apply(parse_timestamp_correct)
)

chargebacks["bank_response_timestamp_clean"] = (
    chargebacks["bank_response_timestamp"]
    .apply(parse_timestamp_correct)
)


print("\nTimestamp cleaning completed.")


# =========================================================
# 4. TIMESTAMP MISSING FLAGS
# =========================================================

chargebacks["transaction_timestamp_missing_flag"] = (
    chargebacks["transaction_timestamp_clean"].isna()
)

chargebacks["report_timestamp_missing_flag"] = (
    chargebacks["report_timestamp_clean"].isna()
)

chargebacks["bank_response_timestamp_missing_flag"] = (
    chargebacks["bank_response_timestamp_clean"].isna()
)


# =========================================================
# 5. CALCULATE DISPUTE REPORTING DELAY
# =========================================================

chargebacks["reporting_delay_hours"] = (
    chargebacks["report_timestamp_clean"]
    -
    chargebacks["transaction_timestamp_clean"]
).dt.total_seconds() / 3600


# Negative reporting delay = data-quality anomaly
chargebacks["negative_reporting_delay_flag"] = (
    chargebacks["reporting_delay_hours"] < 0
)


# =========================================================
# 6. CALCULATE BANK RESPONSE DELAY
# =========================================================

chargebacks["bank_response_delay_hours"] = (
    chargebacks["bank_response_timestamp_clean"]
    -
    chargebacks["report_timestamp_clean"]
).dt.total_seconds() / 3600


# Negative response delay = data-quality anomaly
chargebacks["negative_bank_response_delay_flag"] = (
    chargebacks["bank_response_delay_hours"] < 0
)


# =========================================================
# 7. TIMESTAMP RESULTS
# =========================================================

print("\n========== TIMESTAMP RESULTS ==========")

print(
    "Missing transaction timestamps:",
    chargebacks[
        "transaction_timestamp_missing_flag"
    ].sum()
)

print(
    "Missing report timestamps:",
    chargebacks[
        "report_timestamp_missing_flag"
    ].sum()
)

print(
    "Missing bank response timestamps:",
    chargebacks[
        "bank_response_timestamp_missing_flag"
    ].sum()
)

print(
    "Negative reporting delays:",
    chargebacks[
        "negative_reporting_delay_flag"
    ].sum()
)

print(
    "Negative bank response delays:",
    chargebacks[
        "negative_bank_response_delay_flag"
    ].sum()
)


# =========================================================
# 8. REPORTING DELAY SUMMARY
# =========================================================

print("\n========== REPORTING DELAY SUMMARY ==========")

display(
    chargebacks[
        "reporting_delay_hours"
    ].describe()
)


# =========================================================
# 9. BANK RESPONSE DELAY SUMMARY
# =========================================================

print("\n========== BANK RESPONSE DELAY SUMMARY ==========")

display(
    chargebacks[
        "bank_response_delay_hours"
    ].describe()
)


# =========================================================
# 10. CHECK CORRECTED SAMPLE
# =========================================================

print("\n========== CLEANED CHARGEBACK SAMPLE ==========")

display(
    chargebacks[
        [
            "complaint_id",
            "txn_id",
            "user_id",
            "merchant_id",
            "disputed_amount",
            "disputed_amount_clean",
            "transaction_timestamp",
            "transaction_timestamp_clean",
            "reported_timestamp",
            "report_timestamp_clean",
            "reporting_delay_hours"
        ]
    ].head(15)
)


print("\n========== STEP 22 COMPLETE ==========")

========== STEP 22: CLEANING CHARGEBACK AMOUNTS & TIMESTAMPS ==========

Disputed amount cleaning completed.

========== DISPUTED AMOUNT SUMMARY ==========


count     2701.000000
mean      3045.486109
std       3813.384614
min         50.000000
25%        913.380000
50%       1819.680000
75%       3670.720000
max      45384.610000
Name: disputed_amount_clean, dtype: float64

Missing/invalid disputed amounts: 183

Timestamp cleaning completed.

========== TIMESTAMP RESULTS ==========
Missing transaction timestamps: 235
Missing report timestamps: 209
Missing bank response timestamps: 718
Negative reporting delays: 94
Negative bank response delays: 16

========== REPORTING DELAY SUMMARY ==========


count    2461.000000
mean      154.362510
std       222.463080
min      -128.310833
25%        43.194444
50%        72.000000
75%       168.000000
max      1120.678889
Name: reporting_delay_hours, dtype: float64


========== BANK RESPONSE DELAY SUMMARY ==========


count    2007.000000
mean      354.834052
std       214.777916
min       -20.020556
25%       168.000000
50%       360.000000
75%       551.666389
max       741.205556
Name: bank_response_delay_hours, dtype: float64


========== CLEANED CHARGEBACK SAMPLE ==========


,complaint_id,txn_id,user_id,merchant_id,disputed_amount,disputed_amount_clean,transaction_timestamp,transaction_timestamp_clean,reported_timestamp,report_timestamp_clean,reporting_delay_hours
0,CBK0002082,TXN00004325,USR97580,MCH1127,,NaN,2026/01/28,2026-01-28 00:00:00,02-01-2026,2026-02-01 00:00:00,96.000000
1,CBK0001941,TXN00003720,USR54113,MCH3835,414.69,414.69,25/02/2026 10:24 AM,2026-02-25 10:24:00,1772691855,2026-03-05 06:24:15,188.004167
2,CBK0001799,TXN00012539,USR17980,MCH3700,"Rs. 7,039",7039.00,2026/02/04,2026-02-04 00:00:00,02-05-2026,2026-02-05 00:00:00,24.000000
3,CBK0002465,TXN00017802,USR76148,MCH4534,1303.05,1303.05,30-Mar-2026,2026-03-30 00:00:00,01-Apr-2026,2026-04-01 00:00:00,48.000000
4,CBK0001870,TXN00015944,USR24660,MCH1686,1459.42,1459.42,09-Jan-2026,2026-01-09 00:00:00,1768424501,2026-01-14 21:01:41,141.028056
5,CBK0002663,TXN00009741,USR40631,MCH9584,Rs. 548,548.00,02-05-2026,2026-02-05 00:00:00,02-06-2026,2026-02-06 00:00:00,24.000000
6,CBK0000782,TXN00015086,USR58789,MCH6008,"₹1,949.60",1949.60,05/02/2026,2026-02-05 00:00:00,02-10-2026,2026-02-10 00:00:00,120.000000
7,CBK0001838,TXN00014426,USR30157,MCH4526,4193.52,4193.52,2026-03-26 10:17:08,2026-03-26 10:17:08,2026-04-06 04:17:08,2026-04-06 04:17:08,258.000000
8,CBK0001095,TXN00015316,USR86750,MCH5980,930.83,930.83,2026/01/12,2026-01-12 00:00:00,13/01/2026,2026-01-13 00:00:00,24.000000
9,CBK0002541,TXN00002822,USR11987,MCH3992,,NaN,2026-03-18 10:04:06,2026-03-18 10:04:06,23/03/2026,2026-03-23 00:00:00,109.931667



========== STEP 22 COMPLETE ==========


In [60]:
print("========== STEP 23: FINAL CHARGEBACK CLEANING & VALIDATION ==========")

# =========================================================
# 1. PRESERVE DUPLICATE INFORMATION
# =========================================================

chargebacks["duplicate_complaint_flag"] = (
    chargebacks.duplicated(
        subset=["complaint_id"],
        keep=False
    )
)

print(
    "Rows belonging to duplicate complaint IDs:",
    chargebacks["duplicate_complaint_flag"].sum()
)


# =========================================================
# 2. REMOVE EXACT DUPLICATE ROWS
# =========================================================

before_dedup = len(chargebacks)

chargebacks = chargebacks.drop_duplicates().copy()

after_dedup = len(chargebacks)

print("Exact duplicate rows removed:", before_dedup - after_dedup)
print("Chargeback rows after exact deduplication:", after_dedup)


# =========================================================
# 3. CHECK DUPLICATE COMPLAINT IDs
# =========================================================

duplicate_complaints = (
    chargebacks["complaint_id"]
    .duplicated()
    .sum()
)

print(
    "Remaining duplicate complaint IDs:",
    duplicate_complaints
)


# =========================================================
# 4. CREATE AMOUNT QUALITY FLAG
# =========================================================

chargebacks["amount_quality_flag"] = np.where(
    chargebacks["disputed_amount_clean"].isna(),
    "MISSING_OR_INVALID",
    "VALID"
)


# =========================================================
# 5. CREATE TIMESTAMP QUALITY FLAG
# =========================================================

chargebacks["timestamp_quality_flag"] = np.select(
    [
        chargebacks["transaction_timestamp_clean"].isna(),
        chargebacks["report_timestamp_clean"].isna(),
        chargebacks["bank_response_timestamp_clean"].isna(),
        chargebacks["negative_reporting_delay_flag"],
        chargebacks["negative_bank_response_delay_flag"]
    ],
    [
        "TRANSACTION_TIMESTAMP_MISSING",
        "REPORT_TIMESTAMP_MISSING",
        "BANK_RESPONSE_TIMESTAMP_MISSING",
        "NEGATIVE_REPORTING_DELAY",
        "NEGATIVE_BANK_RESPONSE_DELAY"
    ],
    default="VALID"
)


# =========================================================
# 6. IDENTIFIER VALIDATION
# =========================================================

print("\n========== IDENTIFIER VALIDATION ==========")

for col in [
    "complaint_id",
    "txn_id",
    "user_id",
    "merchant_id"
]:

    missing_count = chargebacks[col].isna().sum()

    blank_count = (
        chargebacks[col]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    print(
        f"{col}: missing={missing_count}, blank={blank_count}"
    )


# =========================================================
# 7. FINAL CHARGEBACK SUMMARY
# =========================================================

print("\n========== FINAL CHARGEBACK SUMMARY ==========")

print("Total rows:", len(chargebacks))

print(
    "Unique complaint IDs:",
    chargebacks["complaint_id"].nunique()
)

print(
    "Unique transaction IDs:",
    chargebacks["txn_id"].nunique()
)

print(
    "Unique user IDs:",
    chargebacks["user_id"].nunique()
)

print(
    "Unique merchant IDs:",
    chargebacks["merchant_id"].nunique()
)


# =========================================================
# 8. AMOUNT QUALITY
# =========================================================

print("\n========== AMOUNT QUALITY ==========")

print(
    "Valid disputed amounts:",
    chargebacks["disputed_amount_clean"].notna().sum()
)

print(
    "Missing/invalid disputed amounts:",
    chargebacks["disputed_amount_clean"].isna().sum()
)


# =========================================================
# 9. TIMESTAMP QUALITY
# =========================================================

print("\n========== TIMESTAMP QUALITY ==========")

print(
    "Valid transaction timestamps:",
    chargebacks["transaction_timestamp_clean"].notna().sum()
)

print(
    "Valid report timestamps:",
    chargebacks["report_timestamp_clean"].notna().sum()
)

print(
    "Valid bank response timestamps:",
    chargebacks["bank_response_timestamp_clean"].notna().sum()
)

print(
    "Negative reporting delays:",
    chargebacks["negative_reporting_delay_flag"].sum()
)

print(
    "Negative bank response delays:",
    chargebacks["negative_bank_response_delay_flag"].sum()
)


# =========================================================
# 10. FIND THE ACTUAL CLEANED COLUMN NAMES
# =========================================================

print("\n========== AVAILABLE CLEANED COLUMNS ==========")

cleaned_columns = [
    col for col in chargebacks.columns
    if any(
        word in col.lower()
        for word in [
            "reason",
            "severity",
            "resolution",
            "channel"
        ]
    )
]

print(cleaned_columns)


# =========================================================
# 11. REASON DISTRIBUTION
# =========================================================

print("\n========== CHARGEBACK REASON DISTRIBUTION ==========")

if "reason_group" in chargebacks.columns:

    display(
        chargebacks["reason_group"]
        .value_counts(dropna=False)
    )

elif "reason_code" in chargebacks.columns:

    display(
        chargebacks["reason_code"]
        .value_counts(dropna=False)
    )

else:

    print("No reason column found.")


# =========================================================
# 12. SEVERITY DISTRIBUTION
# =========================================================

print("\n========== CHARGEBACK SEVERITY DISTRIBUTION ==========")

if "severity_clean" in chargebacks.columns:

    display(
        chargebacks["severity_clean"]
        .value_counts(dropna=False)
    )

elif "severity" in chargebacks.columns:

    display(
        chargebacks["severity"]
        .value_counts(dropna=False)
    )

else:

    print("No severity column found.")


# =========================================================
# 13. RESOLUTION STATUS DISTRIBUTION
# =========================================================

print("\n========== RESOLUTION STATUS DISTRIBUTION ==========")

if "resolution_status_clean" in chargebacks.columns:

    display(
        chargebacks["resolution_status_clean"]
        .value_counts(dropna=False)
    )

elif "resolution_status" in chargebacks.columns:

    display(
        chargebacks["resolution_status"]
        .value_counts(dropna=False)
    )

else:

    print("No resolution status column found.")


# =========================================================
# 14. CHANNEL DISTRIBUTION
# =========================================================

print("\n========== CHARGEBACK CHANNEL DISTRIBUTION ==========")

if "channel_clean" in chargebacks.columns:

    display(
        chargebacks["channel_clean"]
        .value_counts(dropna=False)
    )

elif "channel" in chargebacks.columns:

    display(
        chargebacks["channel"]
        .value_counts(dropna=False)
    )

else:

    print("No channel column found.")


# =========================================================
# 15. SAVE CLEAN CHARGEBACK DATA
# =========================================================

import os

processed_folder = r"../data/processed"

os.makedirs(
    processed_folder,
    exist_ok=True
)

chargeback_output = (
    r"../data/processed/chargebacks_clean.csv"
)

chargebacks.to_csv(
    chargeback_output,
    index=False
)

print("\nSaved clean chargeback dataset:")
print(chargeback_output)

print("\n========== STEP 23 COMPLETE ==========")

========== STEP 23: FINAL CHARGEBACK CLEANING & VALIDATION ==========
Rows belonging to duplicate complaint IDs: 0
Exact duplicate rows removed: 0
Chargeback rows after exact deduplication: 2800
Remaining duplicate complaint IDs: 0

========== IDENTIFIER VALIDATION ==========
complaint_id: missing=0, blank=0
txn_id: missing=0, blank=77
user_id: missing=0, blank=0
merchant_id: missing=0, blank=0

========== FINAL CHARGEBACK SUMMARY ==========
Total rows: 2800
Unique complaint IDs: 2800
Unique transaction IDs: 2568
Unique user IDs: 2294
Unique merchant IDs: 1855

========== AMOUNT QUALITY ==========
Valid disputed amounts: 2621
Missing/invalid disputed amounts: 179

========== TIMESTAMP QUALITY ==========
Valid transaction timestamps: 2570
Valid report timestamps: 2597
Valid bank response timestamps: 2099
Negative reporting delays: 92
Negative bank response delays: 16

========== AVAILABLE CLEANED COLUMNS ==========
['reason_code', 'resolution_status', 'severity', 'channel', 'reason_grou

reason_group
FRAUD_OR_UNAUTHORIZED    959
SERVICE_OR_DELIVERY      704
CUSTOMER_DISPUTE         376
AMOUNT_ISSUE             326
DUPLICATE_PAYMENT        265
DUP DEBIT                 87
NOT DONE BY ME            83
Name: count, dtype: int64


========== CHARGEBACK SEVERITY DISTRIBUTION ==========


severity
MEDIUM     1185
LOW         959
HIGH        505
UNKNOWN     151
Name: count, dtype: int64


========== RESOLUTION STATUS DISTRIBUTION ==========


resolution_status
IN_PROGRESS    599
PENDING        458
REJECTED       443
CLOSED         442
OPEN           435
RESOLVED       423
Name: count, dtype: int64


========== CHARGEBACK CHANNEL DISTRIBUTION ==========


channel
IVR            709
CHATBOT        698
EMAIL          375
BRANCH         366
APP            344
CALL_CENTER    308
Name: count, dtype: Int64


Saved clean chargeback dataset:
../data/processed/chargebacks_clean.csv

========== STEP 23 COMPLETE ==========


In [62]:
print("========== STEP 24A: SAVE CLEANED DATASETS ==========")

import os

# Create processed folder
processed_folder = r"../data/processed"

os.makedirs(
    processed_folder,
    exist_ok=True
)


# =========================================================
# 1. SAVE TRANSACTIONS
# =========================================================

transactions.to_csv(
    r"../data/processed/transactions_clean.csv",
    index=False
)

print(
    "Transactions saved:",
    transactions.shape
)


# =========================================================
# 2. SAVE KYC
# =========================================================

kyc.to_csv(
    r"../data/processed/kyc_clean.csv",
    index=False
)

print(
    "KYC saved:",
    kyc.shape
)


# =========================================================
# 3. SAVE MERCHANTS
# =========================================================

merchants.to_csv(
    r"../data/processed/merchants_clean.csv",
    index=False
)

print(
    "Merchants saved:",
    merchants.shape
)


# =========================================================
# 4. CHARGEBACKS ALREADY SAVED
# =========================================================

chargebacks.to_csv(
    r"../data/processed/chargebacks_clean.csv",
    index=False
)

print(
    "Chargebacks saved:",
    chargebacks.shape
)


print("\n========== FILES SAVED ==========")

print("✓ transactions_clean.csv")
print("✓ kyc_clean.csv")
print("✓ merchants_clean.csv")
print("✓ chargebacks_clean.csv")

print("\n========== STEP 24A COMPLETE ==========")

========== STEP 24A: SAVE CLEANED DATASETS ==========
Transactions saved: (20400, 8)
KYC saved: (36400, 12)
Merchants saved: (6210, 14)
Chargebacks saved: (2800, 29)

========== FILES SAVED ==========
✓ transactions_clean.csv
✓ kyc_clean.csv
✓ merchants_clean.csv
✓ chargebacks_clean.csv

========== STEP 24A COMPLETE ==========


In [63]:
print("========== STEP 24B: CHECK CURRENT DATAFRAMES ==========")

print("\nTransactions:")
print(transactions.shape)
print(transactions.columns.tolist())

print("\nKYC:")
print(kyc.shape)
print(kyc.columns.tolist())

print("\nMerchants:")
print(merchants.shape)
print(merchants.columns.tolist())

print("\nChargebacks:")
print(chargebacks.shape)
print(chargebacks.columns.tolist())

print("\n========== CHECK COMPLETE ==========")

========== STEP 24B: CHECK CURRENT DATAFRAMES ==========

Transactions:
(20400, 8)
['txn_id', 'timestamp', 'user_id', 'merchant_id', 'amount', 'utr', 'mcc', 'status']

KYC:
(36400, 12)
['user_id', 'full_name', 'pan', 'aadhaar', 'date_of_birth', 'city', 'state', 'monthly_income', 'occupation', 'signup_timestamp', 'kyc_status', 'risk_segment']

Merchants:
(6210, 14)
['merchant_id', 'merchant_name', 'mcc', 'merchant_category', 'business_type', 'city', 'state', 'onboarding_date', 'settlement_account', 'merchant_status', 'declared_avg_ticket_size', 'mcc_valid_flag', 'onboarding_date_missing_flag', 'ticket_size_missing_flag']

Chargebacks:
(2800, 29)
['complaint_id', 'txn_id', 'user_id', 'merchant_id', 'transaction_timestamp', 'reported_timestamp', 'disputed_amount', 'reason_code', 'complaint_text', 'resolution_status', 'bank_response_timestamp', 'severity', 'channel', 'reason_group', 'disputed_amount_clean', 'disputed_amount_missing_flag', 'transaction_timestamp_clean', 'report_timestamp_cl

In [65]:
print("========== STEP 24C: FIND CLEANED DATAFRAMES ==========")

# Show all DataFrame variables currently stored in the notebook
for name, obj in globals().items():
    if isinstance(obj, pd.DataFrame):
        print(f"{name:30s} -> {obj.shape}")

========== STEP 24C: FIND CLEANED DATAFRAMES ==========
transactions                   -> (20400, 8)
kyc                            -> (36400, 12)
merchants                      -> (6210, 14)
chargebacks                    -> (2800, 29)
transactions_clean             -> (20000, 16)
kyc_clean                      -> (36122, 21)
merchants_clean                -> (6198, 11)
chargebacks_clean              -> (2800, 13)
duplicate_summary              -> (4, 4)
identity_check                 -> (29355, 5)
identity_profile               -> (29355, 12)
merchant_profile               -> (4480, 26)


In [66]:
print("========== STEP 24D: SAVE FINAL CLEANED DATASETS ==========")

import os

processed_folder = r"../data/processed"
os.makedirs(processed_folder, exist_ok=True)


# =========================================================
# SAVE ACTUAL CLEANED DATAFRAMES
# =========================================================

transactions_clean.to_csv(
    r"../data/processed/transactions_clean.csv",
    index=False
)

kyc_clean.to_csv(
    r"../data/processed/kyc_clean.csv",
    index=False
)

merchants_clean.to_csv(
    r"../data/processed/merchants_clean.csv",
    index=False
)

chargebacks_clean.to_csv(
    r"../data/processed/chargebacks_clean.csv",
    index=False
)


# =========================================================
# VERIFY
# =========================================================

print("\n========== FINAL FILE VERIFICATION ==========")

print(
    "Transactions:",
    transactions_clean.shape,
    "✓"
)

print(
    "KYC:",
    kyc_clean.shape,
    "✓"
)

print(
    "Merchants:",
    merchants_clean.shape,
    "✓"
)

print(
    "Chargebacks:",
    chargebacks_clean.shape,
    "✓"
)


print("\nFiles saved to:")
print(os.path.abspath(processed_folder))

print("\n========== STEP 24D COMPLETE ==========")

========== STEP 24D: SAVE FINAL CLEANED DATASETS ==========

========== FINAL FILE VERIFICATION ==========
Transactions: (20000, 16) ✓
KYC: (36122, 21) ✓
Merchants: (6198, 11) ✓
Chargebacks: (2800, 13) ✓

Files saved to:
C:\Users\VIKASH\Desktop\UPI-Fraud-Ring-Merchant-Analytics\data\processed

========== STEP 24D COMPLETE ==========


In [67]:
print("========== STEP 25: JOIN VALIDATION ==========")

import pandas as pd
import numpy as np


# =========================================================
# 1. PREPARE UNIQUE MASTER IDS
# =========================================================

kyc_users = set(
    kyc_clean["user_id"]
    .dropna()
    .astype(str)
    .str.strip()
)

merchant_ids = set(
    merchants_clean["merchant_id"]
    .dropna()
    .astype(str)
    .str.strip()
)

transaction_ids = set(
    transactions_clean["txn_id"]
    .dropna()
    .astype(str)
    .str.strip()
)


# =========================================================
# 2. TRANSACTION → KYC
# =========================================================

txn_user_ids = (
    transactions_clean["user_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)

transactions_clean["kyc_match_flag"] = (
    txn_user_ids.isin(kyc_users)
)

txn_kyc_match_rate = (
    transactions_clean["kyc_match_flag"].mean() * 100
)


# =========================================================
# 3. TRANSACTION → MERCHANT
# =========================================================

txn_merchant_ids = (
    transactions_clean["merchant_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)

transactions_clean["merchant_match_flag"] = (
    txn_merchant_ids.isin(merchant_ids)
)

txn_merchant_match_rate = (
    transactions_clean["merchant_match_flag"].mean() * 100
)


# =========================================================
# 4. CHARGEBACK → TRANSACTION
# =========================================================

cb_txn_ids = (
    chargebacks_clean["txn_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)

chargebacks_clean["transaction_match_flag"] = (
    cb_txn_ids.isin(transaction_ids)
)

cb_transaction_match_rate = (
    chargebacks_clean["transaction_match_flag"].mean() * 100
)


# =========================================================
# 5. CHARGEBACK → KYC
# =========================================================

cb_user_ids = (
    chargebacks_clean["user_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)

chargebacks_clean["kyc_match_flag"] = (
    cb_user_ids.isin(kyc_users)
)

cb_kyc_match_rate = (
    chargebacks_clean["kyc_match_flag"].mean() * 100
)


# =========================================================
# 6. CHARGEBACK → MERCHANT
# =========================================================

cb_merchant_ids = (
    chargebacks_clean["merchant_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)

chargebacks_clean["merchant_match_flag"] = (
    cb_merchant_ids.isin(merchant_ids)
)

cb_merchant_match_rate = (
    chargebacks_clean["merchant_match_flag"].mean() * 100
)


# =========================================================
# 7. DISPLAY JOIN RESULTS
# =========================================================

print("\n========== JOIN MATCH RATES ==========")

print(
    f"Transaction → KYC:       {txn_kyc_match_rate:.2f}%"
)

print(
    f"Transaction → Merchant:  {txn_merchant_match_rate:.2f}%"
)

print(
    f"Chargeback → Transaction: {cb_transaction_match_rate:.2f}%"
)

print(
    f"Chargeback → KYC:        {cb_kyc_match_rate:.2f}%"
)

print(
    f"Chargeback → Merchant:   {cb_merchant_match_rate:.2f}%"
)


# =========================================================
# 8. DISPLAY MATCH / UNMATCHED COUNTS
# =========================================================

print("\n========== MATCH COUNTS ==========")

print(
    "Transaction → KYC:"
)

print(
    "  Matched:",
    transactions_clean["kyc_match_flag"].sum()
)

print(
    "  Unmatched:",
    (~transactions_clean["kyc_match_flag"]).sum()
)


print(
    "\nTransaction → Merchant:"
)

print(
    "  Matched:",
    transactions_clean["merchant_match_flag"].sum()
)

print(
    "  Unmatched:",
    (~transactions_clean["merchant_match_flag"]).sum()
)


print(
    "\nChargeback → Transaction:"
)

print(
    "  Matched:",
    chargebacks_clean["transaction_match_flag"].sum()
)

print(
    "  Unmatched:",
    (~chargebacks_clean["transaction_match_flag"]).sum()
)


print(
    "\nChargeback → KYC:"
)

print(
    "  Matched:",
    chargebacks_clean["kyc_match_flag"].sum()
)

print(
    "  Unmatched:",
    (~chargebacks_clean["kyc_match_flag"]).sum()
)


print(
    "\nChargeback → Merchant:"
)

print(
    "  Matched:",
    chargebacks_clean["merchant_match_flag"].sum()
)

print(
    "  Unmatched:",
    (~chargebacks_clean["merchant_match_flag"]).sum()
)


print("\n========== STEP 25 COMPLETE ==========")

========== STEP 25: JOIN VALIDATION ==========

========== JOIN MATCH RATES ==========
Transaction → KYC:       31.18%
Transaction → Merchant:  46.38%
Chargeback → Transaction: 93.11%
Chargeback → KYC:        28.82%
Chargeback → Merchant:   42.82%

========== MATCH COUNTS ==========
Transaction → KYC:
  Matched: 6235
  Unmatched: 13765

Transaction → Merchant:
  Matched: 9275
  Unmatched: 10725

Chargeback → Transaction:
  Matched: 2607
  Unmatched: 193

Chargeback → KYC:
  Matched: 807
  Unmatched: 1993

Chargeback → Merchant:
  Matched: 1199
  Unmatched: 1601

========== STEP 25 COMPLETE ==========


In [68]:
print("========== STEP 26: BUILD ENRICHED TRANSACTION FACT TABLE ==========")

import pandas as pd
import numpy as np


# =========================================================
# 1. START WITH TRANSACTIONS
# =========================================================

fact_transactions = transactions_clean.copy()

print(
    "Starting transaction rows:",
    len(fact_transactions)
)


# =========================================================
# 2. PREPARE KYC MASTER
# =========================================================

kyc_master = kyc_clean.copy()

# Keep only useful KYC columns for the transaction fact table
kyc_columns = [
    "user_id",
    "full_name",
    "pan",
    "aadhaar",
    "date_of_birth",
    "city",
    "state",
    "monthly_income",
    "occupation",
    "kyc_status",
    "risk_segment"
]

kyc_master = kyc_master[
    [col for col in kyc_columns if col in kyc_master.columns]
].copy()


# ---------------------------------------------------------
# IMPORTANT:
# KYC may contain multiple records for the same user.
# We need ONE record per user before merging.
# ---------------------------------------------------------

kyc_master = (
    kyc_master
    .drop_duplicates(subset=["user_id"], keep="first")
)


print(
    "Unique KYC users available:",
    kyc_master["user_id"].nunique()
)


# =========================================================
# 3. MERGE KYC INTO TRANSACTIONS
# =========================================================

fact_transactions = fact_transactions.merge(
    kyc_master,
    on="user_id",
    how="left",
    suffixes=("", "_kyc")
)


print(
    "Rows after KYC merge:",
    len(fact_transactions)
)


# =========================================================
# 4. PREPARE MERCHANT MASTER
# =========================================================

merchant_master = merchants_clean.copy()

merchant_columns = [
    "merchant_id",
    "merchant_name",
    "mcc",
    "merchant_category",
    "business_type",
    "city",
    "state",
    "onboarding_date",
    "settlement_account",
    "merchant_status",
    "declared_avg_ticket_size"
]

merchant_master = merchant_master[
    [
        col
        for col in merchant_columns
        if col in merchant_master.columns
    ]
].copy()


# ---------------------------------------------------------
# ONE RECORD PER MERCHANT
# ---------------------------------------------------------

merchant_master = (
    merchant_master
    .drop_duplicates(
        subset=["merchant_id"],
        keep="first"
    )
)


print(
    "Unique merchants available:",
    merchant_master["merchant_id"].nunique()
)


# =========================================================
# 5. MERGE MERCHANT DATA
# =========================================================

fact_transactions = fact_transactions.merge(
    merchant_master,
    on="merchant_id",
    how="left",
    suffixes=("", "_merchant")
)


print(
    "Rows after merchant merge:",
    len(fact_transactions)
)


# =========================================================
# 6. CREATE FINAL MATCH FLAGS
# =========================================================

fact_transactions["kyc_match_flag"] = (
    fact_transactions["user_id"]
    .fillna("")
    .astype(str)
    .str.strip()
    .isin(
        set(
            kyc_master["user_id"]
            .dropna()
            .astype(str)
            .str.strip()
        )
    )
)

fact_transactions["merchant_match_flag"] = (
    fact_transactions["merchant_id"]
    .fillna("")
    .astype(str)
    .str.strip()
    .isin(
        set(
            merchant_master["merchant_id"]
            .dropna()
            .astype(str)
            .str.strip()
        )
    )
)


# =========================================================
# 7. CREATE DATA QUALITY FLAGS
# =========================================================

fact_transactions["user_id_missing_flag"] = (
    fact_transactions["user_id"]
    .isna()
    |
    fact_transactions["user_id"]
    .astype(str)
    .str.strip()
    .eq("")
)

fact_transactions["merchant_id_missing_flag"] = (
    fact_transactions["merchant_id"]
    .isna()
    |
    fact_transactions["merchant_id"]
    .astype(str)
    .str.strip()
    .eq("")
)


# =========================================================
# 8. VERIFY ROW COUNT
# =========================================================

print("\n========== ROW COUNT VALIDATION ==========")

print(
    "Original cleaned transactions:",
    len(transactions_clean)
)

print(
    "Enriched transaction rows:",
    len(fact_transactions)
)

if len(fact_transactions) == len(transactions_clean):
    print("✓ Row count preserved")
else:
    print("⚠ WARNING: Row count changed")


# =========================================================
# 9. VERIFY TRANSACTION ID UNIQUENESS
# =========================================================

print("\n========== TRANSACTION ID VALIDATION ==========")

print(
    "Unique transaction IDs:",
    fact_transactions["txn_id"].nunique()
)

print(
    "Duplicate transaction IDs:",
    fact_transactions["txn_id"].duplicated().sum()
)


# =========================================================
# 10. MATCH SUMMARY
# =========================================================

print("\n========== FINAL MATCH SUMMARY ==========")

print(
    "KYC matched:",
    fact_transactions["kyc_match_flag"].sum()
)

print(
    "KYC unmatched:",
    (~fact_transactions["kyc_match_flag"]).sum()
)

print(
    "Merchant matched:",
    fact_transactions["merchant_match_flag"].sum()
)

print(
    "Merchant unmatched:",
    (~fact_transactions["merchant_match_flag"]).sum()
)


# =========================================================
# 11. DISPLAY IMPORTANT COLUMNS
# =========================================================

print("\n========== ENRICHED DATA SAMPLE ==========")

sample_columns = [
    "txn_id",
    "timestamp",
    "user_id",
    "merchant_id",
    "amount",
    "status",
    "full_name",
    "kyc_status",
    "risk_segment",
    "merchant_name",
    "merchant_category",
    "business_type",
    "merchant_status",
    "kyc_match_flag",
    "merchant_match_flag"
]

sample_columns = [
    col
    for col in sample_columns
    if col in fact_transactions.columns
]

display(
    fact_transactions[sample_columns].head(10)
)


# =========================================================
# 12. SAVE ENRICHED FACT TABLE
# =========================================================

fact_output = (
    r"../data/processed/fact_transactions_enriched.csv"
)

fact_transactions.to_csv(
    fact_output,
    index=False
)

print("\nSaved:")
print(fact_output)

print("\n========== STEP 26 COMPLETE ==========")

========== STEP 26: BUILD ENRICHED TRANSACTION FACT TABLE ==========
Starting transaction rows: 20000
Unique KYC users available: 29355
Rows after KYC merge: 20000
Unique merchants available: 4480
Rows after merchant merge: 20000

========== ROW COUNT VALIDATION ==========
Original cleaned transactions: 20000
Enriched transaction rows: 20000
✓ Row count preserved

========== TRANSACTION ID VALIDATION ==========
Unique transaction IDs: 20000
Duplicate transaction IDs: 0

========== FINAL MATCH SUMMARY ==========
KYC matched: 6235
KYC unmatched: 13765
Merchant matched: 9275
Merchant unmatched: 10725

========== ENRICHED DATA SAMPLE ==========


,txn_id,timestamp,user_id,merchant_id,amount,status,full_name,kyc_status,risk_segment,merchant_name,merchant_category,business_type,merchant_status,kyc_match_flag,merchant_match_flag
0,TXN00011869,2026-01-15 00:11:30,USR45826,MCH7045,15722.34,COMPLETED,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,False,False
1,TXN00010383,2026-01-17 20:09:44,USR79397,MCH5031,6362.90,TXN_FAILED,SANAYA MITTER,VERIFIED,MEDIUM,NaN,NaN,NaN,NaN,True,False
2,TXN00008297,2026-01-23 14:10:16,USR87810,MCH9809,15446.19,S,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,False,False
3,TXN00006448,2026-02-02 20:17:51,USR54287,MCH6928,12110.49,S,<NA>,NaN,<NA>,Tata-Wagle,chemist,PARTNERSHIP,A,False,True
4,TXN00018792,2026-03-31 14:02:37,USR53865,MCH8121,19432.94,TXN_SUCCESS,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,False,False
5,TXN00000400,2026-02-11 01:40:43,USR90546,MCH6773,18866.03,SUCCESS,BIMALA MOHAN,REJECTED,LOW,"parmar, sahota and rajagopalan",Miscellaneous,SOLE-PROPRIETOR,Enabled,True,True
6,TXN00015249,2026-03-10 21:27:31,USR18691,MCH7856,19982.69,COMPLETED,ABDUL DALAL,APPROVED,LOW,"Sood, Dalal and Butala",Hotels,Private Limited,Live,True,True
7,TXN00007121,2026-02-25 00:53:02,USR16629,MCH7753,22687.50,COMPLETED,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,False,False
8,TXN00008460,2026-02-07 09:53:11,USR20899,MCH9111,8256.07,S,UDARSH DUGAL,VERIFIED,LOW,"Dhar, Thaker and Madan",Misc Retail,private_limited,Enabled,True,True
9,TXN00002541,2026-03-30 09:37:49,USR51755,MCH2949,16466.93,FAILED,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,False,False



Saved:
../data/processed/fact_transactions_enriched.csv

========== STEP 26 COMPLETE ==========


In [70]:
print("========== STEP 27: ATTACH CHARGEBACK EVIDENCE ==========")

import pandas as pd
import numpy as np


# =========================================================
# 1. USE THE 29-COLUMN CLEANED CHARGEBACK DATAFRAME
# =========================================================

cb = chargebacks.copy()

print("Chargeback rows:", len(cb))
print("Chargeback columns:", len(cb.columns))


# =========================================================
# 2. KEEP CHARGEBACKS WITH VALID TRANSACTION IDs
# =========================================================

cb_with_txn = cb[
    cb["txn_id"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
].copy()

print(
    "Chargebacks with transaction ID:",
    len(cb_with_txn)
)

print(
    "Chargebacks without transaction ID:",
    len(cb) - len(cb_with_txn)
)


# =========================================================
# 3. CREATE NUMERIC SEVERITY SCORE
# =========================================================

severity_text = (
    cb_with_txn["severity"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.upper()
    .str.strip()
)

severity_text = severity_text.replace({
    "CRITICAL": "HIGH",
    "CRIT": "HIGH",
    "P1": "HIGH",
    "P2": "HIGH",
    "P3": "MEDIUM",
    "P4": "LOW",
    "H": "HIGH",
    "M": "MEDIUM",
    "L": "LOW"
})

severity_score = severity_text.map({
    "HIGH": 3,
    "MEDIUM": 2,
    "LOW": 1,
    "UNKNOWN": 0
})


# =========================================================
# 4. CREATE FRAUD / UNAUTHORIZED FLAG
# =========================================================

fraud_flag = (
    cb_with_txn["reason_group"]
    .fillna("")
    .astype(str)
    .str.upper()
    .eq("FRAUD_OR_UNAUTHORIZED")
)


# =========================================================
# 5. CREATE CHARGEBACK TRANSACTION SUMMARY
# =========================================================

chargeback_txn_summary = (
    cb_with_txn
    .assign(
        severity_score=severity_score,
        fraud_unauthorized_flag=fraud_flag
    )
    .groupby("txn_id")
    .agg(
        chargeback_count=(
            "complaint_id",
            "count"
        ),

        disputed_amount_total=(
            "disputed_amount_clean",
            "sum"
        ),

        max_severity_score=(
            "severity_score",
            "max"
        ),

        fraud_unauthorized_chargeback_count=(
            "fraud_unauthorized_flag",
            "sum"
        ),

        negative_reporting_delay_count=(
            "negative_reporting_delay_flag",
            "sum"
        ),

        negative_bank_response_delay_count=(
            "negative_bank_response_delay_flag",
            "sum"
        )
    )
    .reset_index()
)


# =========================================================
# 6. CONVERT SEVERITY SCORE BACK TO LABEL
# =========================================================

chargeback_txn_summary["max_chargeback_severity"] = (
    chargeback_txn_summary["max_severity_score"]
    .map({
        3: "HIGH",
        2: "MEDIUM",
        1: "LOW",
        0: "UNKNOWN"
    })
    .fillna("UNKNOWN")
)


# Remove temporary score
chargeback_txn_summary = chargeback_txn_summary.drop(
    columns=["max_severity_score"]
)


# =========================================================
# 7. MERGE INTO TRANSACTION FACT TABLE
# =========================================================

fact_transactions = fact_transactions.merge(
    chargeback_txn_summary,
    on="txn_id",
    how="left"
)


# =========================================================
# 8. CREATE CHARGEBACK FLAGS
# =========================================================

fact_transactions["chargeback_flag"] = (
    fact_transactions["chargeback_count"]
    .fillna(0)
    .gt(0)
)

fact_transactions["fraud_chargeback_flag"] = (
    fact_transactions[
        "fraud_unauthorized_chargeback_count"
    ]
    .fillna(0)
    .gt(0)
)


# =========================================================
# 9. FILL MISSING AGGREGATED VALUES
# =========================================================

fact_transactions["chargeback_count"] = (
    fact_transactions["chargeback_count"]
    .fillna(0)
    .astype(int)
)

fact_transactions["disputed_amount_total"] = (
    fact_transactions["disputed_amount_total"]
    .fillna(0)
)

fact_transactions[
    "fraud_unauthorized_chargeback_count"
] = (
    fact_transactions[
        "fraud_unauthorized_chargeback_count"
    ]
    .fillna(0)
    .astype(int)
)

fact_transactions[
    "negative_reporting_delay_count"
] = (
    fact_transactions[
        "negative_reporting_delay_count"
    ]
    .fillna(0)
    .astype(int)
)

fact_transactions[
    "negative_bank_response_delay_count"
] = (
    fact_transactions[
        "negative_bank_response_delay_count"
    ]
    .fillna(0)
    .astype(int)
)

fact_transactions["max_chargeback_severity"] = (
    fact_transactions["max_chargeback_severity"]
    .fillna("NONE")
)


# =========================================================
# 10. ROW COUNT VALIDATION
# =========================================================

print("\n========== ROW COUNT VALIDATION ==========")

print(
    "Rows before chargeback merge:",
    20000
)

print(
    "Rows after chargeback merge:",
    len(fact_transactions)
)

print(
    "Unique transaction IDs:",
    fact_transactions["txn_id"].nunique()
)

print(
    "Duplicate transaction IDs:",
    fact_transactions["txn_id"].duplicated().sum()
)


if (
    len(fact_transactions) == 20000
    and fact_transactions["txn_id"].nunique() == 20000
    and fact_transactions["txn_id"].duplicated().sum() == 0
):
    print("✓ Transaction grain preserved")
else:
    print("⚠ WARNING: Transaction grain changed")


# =========================================================
# 11. CHARGEBACK SUMMARY
# =========================================================

print("\n========== CHARGEBACK TRANSACTION SUMMARY ==========")

print(
    "Transactions with chargebacks:",
    fact_transactions["chargeback_flag"].sum()
)

print(
    "Transactions without chargebacks:",
    (~fact_transactions["chargeback_flag"]).sum()
)

print(
    "Transactions with fraud/unauthorized disputes:",
    fact_transactions["fraud_chargeback_flag"].sum()
)

print(
    "Total disputed amount:",
    round(
        fact_transactions[
            "disputed_amount_total"
        ].sum(),
        2
    )
)


# =========================================================
# 12. CHARGEBACK COUNT DISTRIBUTION
# =========================================================

print("\n========== CHARGEBACK COUNT DISTRIBUTION ==========")

display(
    fact_transactions[
        "chargeback_count"
    ]
    .value_counts()
    .sort_index()
)


# =========================================================
# 13. TOP TRANSACTIONS BY CHARGEBACK COUNT
# =========================================================

print("\n========== TOP TRANSACTIONS BY CHARGEBACK COUNT ==========")

display(
    fact_transactions[
        [
            "txn_id",
            "user_id",
            "merchant_id",
            "amount",
            "chargeback_count",
            "disputed_amount_total",
            "fraud_unauthorized_chargeback_count",
            "max_chargeback_severity",
            "chargeback_flag",
            "fraud_chargeback_flag"
        ]
    ]
    .sort_values(
        [
            "chargeback_count",
            "disputed_amount_total"
        ],
        ascending=False
    )
    .head(10)
)


# =========================================================
# 14. SAVE UPDATED FACT TABLE
# =========================================================

fact_output = (
    r"../data/processed/fact_transactions_enriched.csv"
)

fact_transactions.to_csv(
    fact_output,
    index=False
)

print("\nSaved updated fact table:")
print(fact_output)

print("\n========== STEP 27 COMPLETE ==========")

========== STEP 27: ATTACH CHARGEBACK EVIDENCE ==========
Chargeback rows: 2800
Chargeback columns: 29
Chargebacks with transaction ID: 2723
Chargebacks without transaction ID: 77

========== ROW COUNT VALIDATION ==========
Rows before chargeback merge: 20000
Rows after chargeback merge: 20000
Unique transaction IDs: 20000
Duplicate transaction IDs: 0
✓ Transaction grain preserved

========== CHARGEBACK TRANSACTION SUMMARY ==========
Transactions with chargebacks: 2451
Transactions without chargebacks: 17549
Transactions with fraud/unauthorized disputes: 873
Total disputed amount: 7462256.72

========== CHARGEBACK COUNT DISTRIBUTION ==========


chargeback_count
0    17549
1     2300
2      146
3        5
Name: count, dtype: int64


========== TOP TRANSACTIONS BY CHARGEBACK COUNT ==========


,txn_id,user_id,merchant_id,amount,chargeback_count,disputed_amount_total,fraud_unauthorized_chargeback_count,max_chargeback_severity,chargeback_flag,fraud_chargeback_flag
12538,TXN00008116,USR58627,MCH8540,7018.78,3,22249.40,2,MEDIUM,True,True
8308,TXN00002926,USR30575,MCH1382,22549.49,3,7854.99,1,HIGH,True,True
15102,TXN00011255,USR95217,MCH9572,8219.44,3,5747.65,2,HIGH,True,True
9076,TXN00013107,USR34733,MCH9675,9686.57,3,5623.62,2,HIGH,True,True
1572,TXN00009947,USR58522,MCH5097,8278.87,3,917.78,2,HIGH,True,True
9302,TXN00013078,USR57502,MCH3171,16342.48,2,30866.46,0,MEDIUM,True,False
12702,TXN00014456,USR50579,MCH6621,9295.61,2,18916.34,0,LOW,True,False
5075,TXN00001617,USR71148,MCH1197,24853.54,2,18876.03,0,HIGH,True,False
17072,TXN00010646,USR34897,MCH9352,NaN,2,17261.49,0,LOW,True,False
19827,TXN00001169,USR95219,MCH4897,169.58,2,16524.78,0,MEDIUM,True,False



Saved updated fact table:
../data/processed/fact_transactions_enriched.csv

========== STEP 27 COMPLETE ==========


In [71]:
print("========== STEP 28: TRANSACTION KPIs & FRAUD SIGNALS ==========")

import pandas as pd
import numpy as np


# =========================================================
# 1. ENSURE AMOUNT IS NUMERIC
# =========================================================

fact_transactions["amount_numeric"] = pd.to_numeric(
    fact_transactions["amount"],
    errors="coerce"
)


# =========================================================
# 2. TRANSACTION STATUS STANDARDIZATION
# =========================================================

status = (
    fact_transactions["status"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.upper()
    .str.strip()
)

# Standardize common status variants
status = status.replace({
    "TXN_SUCCESS": "SUCCESS",
    "S": "SUCCESS",
    "COMPLETED": "SUCCESS",

    "TXN_FAILED": "FAILED",
    "F": "FAILED",

    "P": "PENDING"
})

fact_transactions["status_clean"] = status


# =========================================================
# 3. BASIC TRANSACTION FLAGS
# =========================================================

fact_transactions["successful_transaction_flag"] = (
    fact_transactions["status_clean"] == "SUCCESS"
)

fact_transactions["failed_transaction_flag"] = (
    fact_transactions["status_clean"] == "FAILED"
)

fact_transactions["pending_transaction_flag"] = (
    fact_transactions["status_clean"] == "PENDING"
)


# =========================================================
# 4. NEGATIVE AMOUNT FLAG
# =========================================================

fact_transactions["negative_amount_flag"] = (
    fact_transactions["amount_numeric"] < 0
)


fact_transactions["missing_amount_flag"] = (
    fact_transactions["amount_numeric"].isna()
)


# =========================================================
# 5. HIGH-VALUE TRANSACTION FLAG
# =========================================================

# Use the 95th percentile rather than an arbitrary threshold.
valid_amounts = fact_transactions[
    fact_transactions["amount_numeric"] > 0
]["amount_numeric"]

high_value_threshold = valid_amounts.quantile(0.95)

fact_transactions["high_value_transaction_flag"] = (
    fact_transactions["amount_numeric"]
    >= high_value_threshold
)

print(
    "High-value transaction threshold:",
    round(high_value_threshold, 2)
)


# =========================================================
# 6. CREATE TRANSACTION DATE
# =========================================================

fact_transactions["timestamp_clean"] = pd.to_datetime(
    fact_transactions["timestamp"],
    errors="coerce"
)

fact_transactions["transaction_date"] = (
    fact_transactions["timestamp_clean"]
    .dt.date
)

fact_transactions["transaction_hour"] = (
    fact_transactions["timestamp_clean"]
    .dt.hour
)

fact_transactions["transaction_day_of_week"] = (
    fact_transactions["timestamp_clean"]
    .dt.day_name()
)


# =========================================================
# 7. HIGH-RISK KYC FLAG
# =========================================================

if "risk_segment" in fact_transactions.columns:

    fact_transactions["high_risk_user_flag"] = (
        fact_transactions["risk_segment"]
        .fillna("")
        .astype(str)
        .str.upper()
        .str.strip()
        .eq("HIGH")
    )

else:

    fact_transactions["high_risk_user_flag"] = False


# =========================================================
# 8. KYC REJECTION FLAG
# =========================================================

if "kyc_status" in fact_transactions.columns:

    fact_transactions["kyc_rejected_transaction_flag"] = (
        fact_transactions["kyc_status"]
        .fillna("")
        .astype(str)
        .str.upper()
        .str.strip()
        .isin([
            "REJECTED",
            "FAILED"
        ])
    )

else:

    fact_transactions["kyc_rejected_transaction_flag"] = False


# =========================================================
# 9. MERCHANT STATUS RISK FLAG
# =========================================================

if "merchant_status" in fact_transactions.columns:

    fact_transactions["merchant_status_risk_flag"] = (
        fact_transactions["merchant_status"]
        .fillna("")
        .astype(str)
        .str.upper()
        .str.strip()
        .isin([
            "SUSPENDED",
            "INACTIVE"
        ])
    )

else:

    fact_transactions["merchant_status_risk_flag"] = False


# =========================================================
# 10. COMBINED DATA-QUALITY FLAG
# =========================================================

fact_transactions["data_quality_issue_flag"] = (
    fact_transactions["missing_amount_flag"]
    |
    fact_transactions["negative_amount_flag"]
    |
    ~fact_transactions["kyc_match_flag"]
    |
    ~fact_transactions["merchant_match_flag"]
)


# =========================================================
# 11. BASIC KPI SUMMARY
# =========================================================

print("\n========== TRANSACTION KPIs ==========")

print(
    "Total transactions:",
    len(fact_transactions)
)

print(
    "Total transaction value:",
    round(
        fact_transactions[
            "amount_numeric"
        ]
        .clip(lower=0)
        .sum(),
        2
    )
)

print(
    "Average transaction value:",
    round(
        fact_transactions[
            "amount_numeric"
        ]
        .clip(lower=0)
        .mean(),
        2
    )
)

print(
    "Successful transactions:",
    fact_transactions[
        "successful_transaction_flag"
    ].sum()
)

print(
    "Failed transactions:",
    fact_transactions[
        "failed_transaction_flag"
    ].sum()
)

print(
    "Pending transactions:",
    fact_transactions[
        "pending_transaction_flag"
    ].sum()
)

print(
    "Missing amounts:",
    fact_transactions[
        "missing_amount_flag"
    ].sum()
)

print(
    "Negative amounts:",
    fact_transactions[
        "negative_amount_flag"
    ].sum()
)


# =========================================================
# 12. STATUS DISTRIBUTION
# =========================================================

print("\n========== TRANSACTION STATUS ==========")

status_summary = (
    fact_transactions["status_clean"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="transaction_count")
)

status_summary["percentage"] = (
    status_summary["transaction_count"]
    / len(fact_transactions)
    * 100
)

display(status_summary)


# =========================================================
# 13. CHARGEBACK METRICS
# =========================================================

print("\n========== CHARGEBACK KPIs ==========")

total_transactions = len(fact_transactions)

chargeback_transactions = (
    fact_transactions["chargeback_flag"].sum()
)

chargeback_ratio = (
    chargeback_transactions
    / total_transactions
    * 100
)

print(
    "Transactions with chargebacks:",
    chargeback_transactions
)

print(
    "Chargeback-to-transaction ratio:",
    round(chargeback_ratio, 2),
    "%"
)

print(
    "Fraud/unauthorized chargeback transactions:",
    fact_transactions[
        "fraud_chargeback_flag"
    ].sum()
)

print(
    "Total disputed amount:",
    round(
        fact_transactions[
            "disputed_amount_total"
        ].sum(),
        2
    )
)


# =========================================================
# 14. FRAUD SIGNAL SUMMARY
# =========================================================

print("\n========== FRAUD SIGNAL SUMMARY ==========")

signals = {
    "High-value transactions":
        fact_transactions[
            "high_value_transaction_flag"
        ].sum(),

    "Chargeback transactions":
        fact_transactions[
            "chargeback_flag"
        ].sum(),

    "Fraud/unauthorized disputes":
        fact_transactions[
            "fraud_chargeback_flag"
        ].sum(),

    "High-risk users":
        fact_transactions[
            "high_risk_user_flag"
        ].sum(),

    "KYC rejected/failed":
        fact_transactions[
            "kyc_rejected_transaction_flag"
        ].sum(),

    "Risky merchant status":
        fact_transactions[
            "merchant_status_risk_flag"
        ].sum(),

    "Negative amount":
        fact_transactions[
            "negative_amount_flag"
        ].sum(),

    "Missing amount":
        fact_transactions[
            "missing_amount_flag"
        ].sum(),

    "KYC unmatched":
        (~fact_transactions[
            "kyc_match_flag"
        ]).sum(),

    "Merchant unmatched":
        (~fact_transactions[
            "merchant_match_flag"
        ]).sum()
}

signal_summary = pd.DataFrame(
    signals.items(),
    columns=[
        "fraud_signal",
        "transaction_count"
    ]
)

signal_summary["percentage"] = (
    signal_summary["transaction_count"]
    / total_transactions
    * 100
)

display(signal_summary)


# =========================================================
# 15. SAVE UPDATED FACT TABLE
# =========================================================

fact_output = (
    r"../data/processed/fact_transactions_enriched.csv"
)

fact_transactions.to_csv(
    fact_output,
    index=False
)

print("\nSaved updated fact table:")
print(fact_output)

print("\n========== STEP 28 COMPLETE ==========")

========== STEP 28: TRANSACTION KPIs & FRAUD SIGNALS ==========
High-value transaction threshold: 23738.56

========== TRANSACTION KPIs ==========
Total transactions: 20000
Total transaction value: 219327606.06
Average transaction value: 12185.54
Successful transactions: 17053
Failed transactions: 1192
Pending transactions: 512
Missing amounts: 2001
Negative amounts: 420

========== TRANSACTION STATUS ==========


,status,transaction_count,percentage
0,SUCCESS,17053,85.265
1,FAILED,1192,5.960
2,PENDING,512,2.560
3,FAIL,391,1.955
4,DECLINED,372,1.860
5,PROCESSING,260,1.300
6,INITIATED,220,1.100



========== CHARGEBACK KPIs ==========
Transactions with chargebacks: 2451
Chargeback-to-transaction ratio: 12.26 %
Fraud/unauthorized chargeback transactions: 873
Total disputed amount: 7462256.72

========== FRAUD SIGNAL SUMMARY ==========


,fraud_signal,transaction_count,percentage
0,High-value transactions,879,4.395
1,Chargeback transactions,2451,12.255
2,Fraud/unauthorized disputes,873,4.365
3,High-risk users,686,3.430
4,KYC rejected/failed,491,2.455
5,Risky merchant status,738,3.690
6,Negative amount,420,2.100
7,Missing amount,2001,10.005
8,KYC unmatched,13765,68.825
9,Merchant unmatched,10725,53.625



Saved updated fact table:
../data/processed/fact_transactions_enriched.csv

========== STEP 28 COMPLETE ==========
